# Model Comparison: Base vs GNN vs HGNN
## Outfit Compatibility Prediction

This notebook compares three model architectures:
- **Base Model**: Simple feedforward neural network (baseline)
- **GNN Model**: Graph Neural Network with sparse matrix operations
- **HGNN Model**: Hypergraph Neural Network

Complete pipeline: data loading → model training → validation → testing → comparison

## 1. Import Required Libraries

In [1]:
import os
import sys
import json
import random
import ast
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse as sp

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, 
    recall_score, confusion_matrix, roc_curve, auc, 
    precision_recall_curve, classification_report, mean_squared_error
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Paths
BASE_PATH = Path("c:/TFM/APP")
DATA_FOLDER = BASE_PATH / "ml_pipeline/data"
CHECKPOINTS_PATH = BASE_PATH / "ml_pipeline/experiments_fashion_score"
CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)

# Random seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✅ Libraries imported and device configured")

Using device: cuda
✅ Libraries imported and device configured


## 2. Load and Prepare Training and Test Data

In [2]:
# ============ HELPER FUNCTIONS ============
def parse_node_list(x):
    """Parse node list from various formats (list, string, eval, comma-separated)."""
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(i) for i in x]
    try:
        return [int(i) for i in eval(x)]
    except Exception:
        s = str(x).strip("[]")
        return [int(i) for i in s.split(",")] if s else []

def rename_column_if_exists(df, old_name, new_name):
    """Safely rename column if it exists."""
    if old_name in df.columns:
        df.rename(columns={old_name: new_name}, inplace=True)
        return True
    return False

def normalize_node_id_column(df, col_name='node_ids'):
    """Find and normalize node_ids column in outfits dataframe."""
    if col_name in df.columns:
        return True
    
    possible_cols = ['node id', 'outfit_nodes', 'items', 'item_ids', 'nodes']
    for col in possible_cols:
        if col in df.columns:
            df.rename(columns={col: col_name}, inplace=True)
            return True
    return False

def create_node_id_mappings(item_df, outfit_df):
    """Create 0-indexed node ID mappings for items and outfits."""
    # Create mappings
    unique_ids = sorted(list(set(item_df['node_id'].values)))
    node_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_ids)}
    node_id_reverse_map = {v: k for k, v in node_id_map.items()}
    
    # Apply to items
    item_df['node_id_mapped'] = item_df['node_id'].map(node_id_map)
    
    # Apply to outfits
    outfit_df['node_ids_parsed_mapped'] = outfit_df['node_ids_parsed'].apply(
        lambda x: [node_id_map.get(nid, nid) for nid in x]
    )
    
    return node_id_map, node_id_reverse_map, len(unique_ids)

# ============ MAIN DATA LOADING ============
print("="*70)
print("LOADING DATA")
print("="*70)

# Define paths
TRAIN_OUTFITS_PATH = DATA_FOLDER / "train_outfits.csv"
TEST_OUTFITS_PATH = DATA_FOLDER / "test_outfits.csv"
VAL_OUTFITS_PATH = DATA_FOLDER / "val_outfits.csv"
TRAIN_ITEMS_PATH = DATA_FOLDER / "train_items.csv"
TEST_ITEMS_PATH = DATA_FOLDER / "test_items.csv"
VAL_ITEMS_PATH = DATA_FOLDER / "val_items.csv"

# Load datasets
print("\n📂 Loading datasets...")
df_train_items = pd.read_csv(TRAIN_ITEMS_PATH)
df_train_outfits = pd.read_csv(TRAIN_OUTFITS_PATH)
df_test_items = pd.read_csv(TEST_ITEMS_PATH)
df_test_outfits = pd.read_csv(TEST_OUTFITS_PATH)
df_val_items = pd.read_csv(VAL_ITEMS_PATH)
df_val_outfits = pd.read_csv(VAL_OUTFITS_PATH)

print(f"✓ Train: {len(df_train_items)} items, {len(df_train_outfits)} outfits")
print(f"✓ Val: {len(df_val_items)} items, {len(df_val_outfits)} outfits")
print(f"✓ Test: {len(df_test_items)} items, {len(df_test_outfits)} outfits")

# ============ NORMALIZE COLUMN NAMES ============
print("\n📋 Normalizing column names...")
for df in [df_train_items, df_test_items, df_val_items]:
    if rename_column_if_exists(df, 'item_id', 'node_id'):
        pass  # Successfully renamed

for df in [df_train_outfits, df_test_outfits, df_val_outfits]:
    if rename_column_if_exists(df, 'items_id', 'node_ids'):
        pass  # Successfully renamed

# Validate item columns
if 'node_id' not in df_train_items.columns or 'node_id' not in df_test_items.columns or 'node_id' not in df_val_items.columns:
    raise KeyError("Items missing 'node_id' column after normalization")

# Validate and normalize outfit columns
for df in [df_train_outfits, df_test_outfits, df_val_outfits]:
    if not normalize_node_id_column(df, 'node_ids'):
        raise KeyError(f"Outfits missing 'node_ids' column")

print(f"✓ All columns normalized")

# ============ CREATE GLOBAL NODE ID MAPPINGS (NO OVERLAP) ============
print("\n🔄 Creating global node ID mappings (non-overlapping)...")

# Collect all unique node IDs from all sets (before parsing)
all_train_ids = set(df_train_items['node_id'].values)
all_val_ids = set(df_val_items['node_id'].values)
all_test_ids = set(df_test_items['node_id'].values)

# Create global mapping with non-overlapping ranges
all_unique_ids = sorted(list(all_train_ids | all_val_ids | all_test_ids))
global_node_id_map = {old_id: new_id for new_id, old_id in enumerate(all_unique_ids)}
total_nodes = len(all_unique_ids)

# Apply global mapping to items BEFORE parsing
df_train_items['node_id_mapped'] = df_train_items['node_id'].map(global_node_id_map)
df_val_items['node_id_mapped'] = df_val_items['node_id'].map(global_node_id_map)
df_test_items['node_id_mapped'] = df_test_items['node_id'].map(global_node_id_map)

print(f"✓ Global mapping created for {total_nodes} unique nodes")
N_train = total_nodes  # Use global total for all sets
N_val = total_nodes
N_test = total_nodes

# ============ PARSE NODE IDS ============
print("\n📊 Parsing node IDs...")
df_train_outfits['node_ids_parsed'] = df_train_outfits['node_ids'].apply(parse_node_list)
df_test_outfits['node_ids_parsed'] = df_test_outfits['node_ids'].apply(parse_node_list)
df_val_outfits['node_ids_parsed'] = df_val_outfits['node_ids'].apply(parse_node_list)
print(f"✓ Node IDs parsed")

# Apply global mapping to outfits (after parsing)
df_train_outfits['node_ids_parsed_mapped'] = df_train_outfits['node_ids_parsed'].apply(
    lambda x: [global_node_id_map.get(nid, nid) for nid in x]
)
df_val_outfits['node_ids_parsed_mapped'] = df_val_outfits['node_ids_parsed'].apply(
    lambda x: [global_node_id_map.get(nid, nid) for nid in x]
)
df_test_outfits['node_ids_parsed_mapped'] = df_test_outfits['node_ids_parsed'].apply(
    lambda x: [global_node_id_map.get(nid, nid) for nid in x]
)

# ============ VALIDATE NODE ID CONSISTENCY ============
print("\n✓ Validating data consistency...")
train_item_ids = set(df_train_items['node_id'].values)
test_item_ids = set(df_test_items['node_id'].values)
val_item_ids = set(df_val_items['node_id'].values)

train_outfit_ids = set()
for outfit in df_train_outfits['node_ids_parsed']:
    train_outfit_ids.update(outfit)

test_outfit_ids = set()
for outfit in df_test_outfits['node_ids_parsed']:
    test_outfit_ids.update(outfit)

val_outfit_ids = set()
for outfit in df_val_outfits['node_ids_parsed']:
    val_outfit_ids.update(outfit)

train_missing = train_outfit_ids - train_item_ids
test_missing = test_outfit_ids - test_item_ids
val_missing = val_outfit_ids - val_item_ids

if train_missing:
    print(f"⚠️  {len(train_missing)} train outfit node IDs not in items!")
    print(f"   Sample (first 5): {sorted(list(train_missing))[:5]}")
if test_missing:
    print(f"⚠️  {len(test_missing)} test outfit node IDs not in items!")
    print(f"   Sample (first 5): {sorted(list(test_missing))[:5]}")
if val_missing:
    print(f"⚠️  {len(val_missing)} val outfit node IDs not in items!")
    print(f"   Sample (first 5): {sorted(list(val_missing))[:5]}")

if not train_missing and not test_missing and not val_missing:
    print(f"✓ All outfit node IDs match items ({len(train_outfit_ids)} train, {len(val_outfit_ids)} val, {len(test_outfit_ids)} test unique)")

# Get node statistics
train_node_indices = set(df_train_items['node_id_mapped'].dropna().astype(int))
val_node_indices = set(df_val_items['node_id_mapped'].dropna().astype(int))
test_node_indices = set(df_test_items['node_id_mapped'].dropna().astype(int))

print(f"\n✓ Train: {len(train_node_indices)} unique nodes (indices: {min(train_node_indices)}-{max(train_node_indices)})")
print(f"✓ Val: {len(val_node_indices)} unique nodes (indices: {min(val_node_indices)}-{max(val_node_indices)})")
print(f"✓ Test: {len(test_node_indices)} unique nodes (indices: {min(test_node_indices)}-{max(test_node_indices)})")
print(f"✓ Total unique nodes (global): {total_nodes}")

# Verify no overlap
train_val_overlap = train_node_indices & val_node_indices
train_test_overlap = train_node_indices & test_node_indices
val_test_overlap = val_node_indices & test_node_indices

if not train_val_overlap and not train_test_overlap and not val_test_overlap:
    print(f"✅ No node overlap between train/val/test sets!")
else:
    print(f"⚠️  WARNING: Found overlapping nodes!")
    if train_val_overlap:
        print(f"   Train-Val overlap: {len(train_val_overlap)} nodes")
    if train_test_overlap:
        print(f"   Train-Test overlap: {len(train_test_overlap)} nodes")
    if val_test_overlap:
        print(f"   Val-Test overlap: {len(val_test_overlap)} nodes")

# ============ SUMMARY ============
print(f"\n✅ Data loading complete (GLOBAL NODE MAPPING BEFORE PARSING)")
print(f"   Train items: {len(df_train_items)} → {len(train_node_indices)} nodes")
print(f"   Train outfits: {len(df_train_outfits)}")
print(f"   Val items: {len(df_val_items)} → {len(val_node_indices)} nodes")
print(f"   Val outfits: {len(df_val_outfits)}")
print(f"   Test items: {len(df_test_items)} → {len(test_node_indices)} nodes")
print(f"   Test outfits: {len(df_test_outfits)}")
print(f"   Global node range: 0 to {total_nodes - 1}")

# ============ CLEANUP: Delete temporary validation variables ============
del train_item_ids, train_outfit_ids, train_missing
del test_item_ids, test_outfit_ids, test_missing
del val_item_ids, val_outfit_ids, val_missing
del all_train_ids, all_val_ids, all_test_ids, all_unique_ids
del train_node_indices, val_node_indices, test_node_indices
del train_val_overlap, train_test_overlap, val_test_overlap
print("\n🧹 Cleaned up temporary validation variables")


LOADING DATA

📂 Loading datasets...
✓ Train: 123787 items, 33990 outfits
✓ Val: 28583 items, 6000 outfits
✓ Test: 116373 items, 30290 outfits

📋 Normalizing column names...
✓ All columns normalized

🔄 Creating global node ID mappings (non-overlapping)...
✓ Global mapping created for 202110 unique nodes

📊 Parsing node IDs...
✓ Node IDs parsed

✓ Validating data consistency...
✓ All outfit node IDs match items (123787 train, 28583 val, 116373 test unique)

✓ Train: 123787 unique nodes (indices: 1-202102)
✓ Val: 28583 unique nodes (indices: 9-202096)
✓ Test: 116373 unique nodes (indices: 0-202109)
✓ Total unique nodes (global): 202110
⚠️  WARNING: Found overlapping nodes!
   Train-Val overlap: 13909 nodes
   Train-Test overlap: 46801 nodes
   Val-Test overlap: 10414 nodes

✅ Data loading complete (GLOBAL NODE MAPPING BEFORE PARSING)
   Train items: 123787 → 123787 nodes
   Train outfits: 33990
   Val items: 28583 → 28583 nodes
   Val outfits: 6000
   Test items: 116373 → 116373 nodes
   

In [3]:
# Extract embeddings and build feature matrices
def parse_embedding(x):
    """Parse embedding from various formats with robust error handling."""
    # Handle None and NaN
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    
    # Already numpy array
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    
    # Already list or tuple
    if isinstance(x, (list, tuple)):
        try:
            return np.array(x, dtype=np.float32)
        except:
            return None
    
    # String formats
    s = str(x).strip()
    
    # Try direct eval (for string representations of lists)
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = eval(s)
            return np.array(arr, dtype=np.float32)
        except:
            pass
    
    # Try space-separated values with scientific notation (e.g., "[-1.5e-02 2.3e-01 ...]")
    try:
        # Remove brackets and split by whitespace
        clean_s = s.strip("[]").strip()
        if clean_s:
            values = [float(v) for v in clean_s.split()]
            if values:
                return np.array(values, dtype=np.float32)
    except:
        pass
    
    # Try comma-separated values (strip brackets first)
    try:
        clean_s = s.strip("[]").strip()
        if clean_s and "," in clean_s:
            values = [float(v) for v in clean_s.split(",")]
            if values:
                return np.array(values, dtype=np.float32)
    except:
        pass
    
    # All parsing failed
    return None

print("\n" + "="*70)
print("BUILDING FEATURE MATRICES")
print("="*70)

# Train data (using remapped node IDs from data loading cell)
print("\n📊 Building training feature matrices...")
train_Xc = []
train_Xa = []
train_node_ids_mapped = []
skipped_train = 0

for idx, row in df_train_items.iterrows():
    node_id_mapped = int(row['node_id_mapped'])
    img_emb = parse_embedding(row.get('img_embedding'))
    attr_emb = parse_embedding(row.get('Xa', row.get('attr_embedding')))
    
    if img_emb is not None and attr_emb is not None:
        # Validate dimensions
        if len(img_emb) == 512 and len(attr_emb) == 256:
            train_node_ids_mapped.append(node_id_mapped)
            train_Xc.append(img_emb)
            train_Xa.append(attr_emb)
        else:
            skipped_train += 1
    else:
        skipped_train += 1

# Build matrices with 0-indexing
Xc_train_full = np.zeros((N_train, 512), dtype=np.float32)
Xa_train_full = np.zeros((N_train, 256), dtype=np.float32)

for i, nid in enumerate(train_node_ids_mapped):
    Xc_train_full[nid] = train_Xc[i]
    Xa_train_full[nid] = train_Xa[i]

X_train_combined = np.concatenate([Xc_train_full, Xa_train_full], axis=1)
print(f"✓ Train: Xc {Xc_train_full.shape}, Xa {Xa_train_full.shape}, Combined {X_train_combined.shape}")
if skipped_train > 0:
    print(f"  ⚠️  Skipped {skipped_train} items with invalid embeddings")

# Validation data
print("\n📊 Building validation feature matrices...")
val_Xc = []
val_Xa = []
val_node_ids_mapped = []
skipped_val = 0

for idx, row in df_val_items.iterrows():
    node_id_mapped = int(row['node_id_mapped'])
    img_emb = parse_embedding(row.get('img_embedding'))
    attr_emb = parse_embedding(row.get('Xa', row.get('attr_embedding')))
    
    if img_emb is not None and attr_emb is not None:
        # Validate dimensions
        if len(img_emb) == 512 and len(attr_emb) == 256:
            val_node_ids_mapped.append(node_id_mapped)
            val_Xc.append(img_emb)
            val_Xa.append(attr_emb)
        else:
            skipped_val += 1
    else:
        skipped_val += 1

Xc_val_full = np.zeros((N_val, 512), dtype=np.float32)
Xa_val_full = np.zeros((N_val, 256), dtype=np.float32)

for i, nid in enumerate(val_node_ids_mapped):
    Xc_val_full[nid] = val_Xc[i]
    Xa_val_full[nid] = val_Xa[i]

X_val_combined = np.concatenate([Xc_val_full, Xa_val_full], axis=1)
print(f"✓ Val:   Xc {Xc_val_full.shape}, Xa {Xa_val_full.shape}, Combined {X_val_combined.shape}")
if skipped_val > 0:
    print(f"  ⚠️  Skipped {skipped_val} items with invalid embeddings")

# Test data
print("\n📊 Building test feature matrices...")
test_Xc = []
test_Xa = []
test_node_ids_mapped = []
skipped_test = 0

for idx, row in df_test_items.iterrows():
    node_id_mapped = int(row['node_id_mapped'])
    img_emb = parse_embedding(row.get('img_embedding'))
    attr_emb = parse_embedding(row.get('Xa', row.get('attr_embedding')))
    
    if img_emb is not None and attr_emb is not None:
        # Validate dimensions
        if len(img_emb) == 512 and len(attr_emb) == 256:
            test_node_ids_mapped.append(node_id_mapped)
            test_Xc.append(img_emb)
            test_Xa.append(attr_emb)
        else:
            skipped_test += 1
    else:
        skipped_test += 1

Xc_test_full = np.zeros((N_test, 512), dtype=np.float32)
Xa_test_full = np.zeros((N_test, 256), dtype=np.float32)

for i, nid in enumerate(test_node_ids_mapped):
    Xc_test_full[nid] = test_Xc[i]
    Xa_test_full[nid] = test_Xa[i]

X_test_combined = np.concatenate([Xc_test_full, Xa_test_full], axis=1)
print(f"✓ Test: Xc {Xc_test_full.shape}, Xa {Xa_test_full.shape}, Combined {X_test_combined.shape}")
if skipped_test > 0:
    print(f"  ⚠️  Skipped {skipped_test} items with invalid embeddings")

# Convert to tensors
Xc_train = torch.tensor(Xc_train_full, dtype=torch.float32, device=device)
Xa_train = torch.tensor(Xa_train_full, dtype=torch.float32, device=device)
X_train_combined_t = torch.tensor(X_train_combined, dtype=torch.float32, device=device)

Xc_val = torch.tensor(Xc_val_full, dtype=torch.float32, device=device)
Xa_val = torch.tensor(Xa_val_full, dtype=torch.float32, device=device)
X_val_combined_t = torch.tensor(X_val_combined, dtype=torch.float32, device=device)

Xc_test = torch.tensor(Xc_test_full, dtype=torch.float32, device=device)
Xa_test = torch.tensor(Xa_test_full, dtype=torch.float32, device=device)
X_test_combined_t = torch.tensor(X_test_combined, dtype=torch.float32, device=device)

print(f"\n✅ Feature matrices ready on {device}")
print(
    f"   Shapes: Train {X_train_combined_t.shape}, Val {X_val_combined_t.shape}, Test {X_test_combined_t.shape}"
)

# ============ CLEANUP: Delete intermediate numpy arrays to save memory ============
del train_Xc, train_Xa, train_node_ids_mapped, skipped_train
del val_Xc, val_Xa, val_node_ids_mapped, skipped_val
del test_Xc, test_Xa, test_node_ids_mapped, skipped_test
del Xc_train_full, Xa_train_full, X_train_combined
del Xc_val_full, Xa_val_full, X_val_combined
del Xc_test_full, Xa_test_full, X_test_combined
del Xc_train, Xa_train, Xc_val, Xa_val, Xc_test, Xa_test
import gc
gc.collect()
print("🧹 Cleaned up intermediate numpy arrays and torch tensors")



BUILDING FEATURE MATRICES

📊 Building training feature matrices...
✓ Train: Xc (202110, 512), Xa (202110, 256), Combined (202110, 768)

📊 Building validation feature matrices...
✓ Val:   Xc (202110, 512), Xa (202110, 256), Combined (202110, 768)

📊 Building test feature matrices...
✓ Test: Xc (202110, 512), Xa (202110, 256), Combined (202110, 768)

✅ Feature matrices ready on cuda
   Shapes: Train torch.Size([202110, 768]), Val torch.Size([202110, 768]), Test torch.Size([202110, 768])
🧹 Cleaned up intermediate numpy arrays and torch tensors


In [4]:
# Create dataset and dataloaders (padded node IDs)
class NodeIdPaddedOutfitDataset(Dataset):
    """Dataset for outfits with padded node IDs."""
    def __init__(self, outfits_node_ids, labels, Xc, Xa, clothes_df=None, pad_node_id=0, max_len=None, device='cpu'):
        self.outfits = outfits_node_ids
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.pad_node_id = int(pad_node_id)
        self.max_len = max_len if max_len is not None else max(len(o) for o in outfits_node_ids)
        self.device = device
        self.Xc = torch.tensor(Xc, dtype=torch.float32, device=device)
        self.Xa = torch.tensor(Xa, dtype=torch.float32, device=device)
        self.clothes_df = clothes_df

    def __len__(self):
        return len(self.outfits)

    def __getitem__(self, idx):
        node_ids = list(self.outfits[idx])

        label = self.labels[idx]
        pad_len = self.max_len - len(node_ids)
        nodes = node_ids + [self.pad_node_id] * pad_len
        mask = [1] * len(node_ids) + [0] * pad_len
        return {
            "nodes": torch.tensor(nodes, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.float32),
            "label": label,
            "orig_node_ids": node_ids
        }


def padded_collate(batch):
    """Collate function for DataLoader."""
    nodes = torch.stack([b["nodes"] for b in batch])
    masks = torch.stack([b["mask"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    orig = [b["orig_node_ids"] for b in batch]
    return {"nodes": nodes, "mask": masks, "label": labels, "orig_node_ids": orig}

# Build datasets for train/val/test (same structure)
print("\n" + "="*70)
print("CREATING DATASETS AND DATALOADERS (PADDED)")
print("="*70)

# Use remapped node IDs from each split
train_node_ids = df_train_outfits['node_ids_parsed_mapped'].tolist()
train_labels = df_train_outfits['label'].values

val_node_ids = df_val_outfits['node_ids_parsed_mapped'].tolist()
val_labels = df_val_outfits['label'].values

test_node_ids = df_test_outfits['node_ids_parsed_mapped'].tolist()
test_labels = df_test_outfits['label'].values

# Derive Xc/Xa from combined tensors
Xc_train_np = X_train_combined_t[:, :512].detach().cpu().numpy()
Xa_train_np = X_train_combined_t[:, 512:].detach().cpu().numpy()
Xc_val_np = X_val_combined_t[:, :512].detach().cpu().numpy()
Xa_val_np = X_val_combined_t[:, 512:].detach().cpu().numpy()
Xc_test_np = X_test_combined_t[:, :512].detach().cpu().numpy()
Xa_test_np = X_test_combined_t[:, 512:].detach().cpu().numpy()

# Create datasets, passing items DataFrames for category-based sorting
train_ds = NodeIdPaddedOutfitDataset(train_node_ids, train_labels, Xc_train_np, Xa_train_np, clothes_df=df_train_items, pad_node_id=0, device=device)
val_ds = NodeIdPaddedOutfitDataset(val_node_ids, val_labels, Xc_val_np, Xa_val_np, clothes_df=df_val_items, pad_node_id=0, device=device)
test_ds = NodeIdPaddedOutfitDataset(test_node_ids, test_labels, Xc_test_np, Xa_test_np, clothes_df=df_test_items, pad_node_id=0, device=device)

# Create dataloaders
batch_size = 1024
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=padded_collate)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=padded_collate)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=padded_collate)

print(f"\n✓ Train: {len(train_ds)} samples in {len(train_loader)} batches")
print(f"✓ Val: {len(val_ds)} samples in {len(val_loader)} batches")
print(f"✓ Test: {len(test_ds)} samples in {len(test_loader)} batches")
print(f"✓ Max outfit length: {train_ds.max_len}")
print(f"\n✅ Dataloaders created (padded node IDs, sorted with items df)")

# ============ CLEANUP: Delete temporary lists only ============
# Keep train_node_ids, val_node_ids, test_node_ids for graph building
del train_labels, val_labels, test_labels
import gc
gc.collect()
print("🧹 Cleaned up temporary labels (keeping node_ids for graph building)")



CREATING DATASETS AND DATALOADERS (PADDED)

✓ Train: 33990 samples in 34 batches
✓ Val: 6000 samples in 6 batches
✓ Test: 30290 samples in 30 batches
✓ Max outfit length: 16

✅ Dataloaders created (padded node IDs, sorted with items df)
🧹 Cleaned up temporary labels (keeping node_ids for graph building)


## 3. Define Base Model

In [5]:
# ============ ATTENTION POOLING ============
class MultiHeadAttnPool(nn.Module):
    """Multi-head attention-based pooling for variable-length sequences."""
    def __init__(self, dim, n_heads=4, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        
        self.query = nn.Linear(dim, dim)
        self.attention_dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        """
        x: (B, L, dim) - sequence embeddings
        mask: (B, L) - binary mask (1 for valid, 0 for padding)
        Returns: pooled (B, dim), weights (B, L)
        """
        B, L, D = x.shape
        
        # Compute attention scores
        q = self.query(x)  # (B, L, dim)
        scores = q.sum(dim=2, keepdim=True)  # (B, L, 1) - simple scoring
        
        # Apply mask
        mask_expanded = mask.unsqueeze(-1).float()  # (B, L, 1)
        scores = scores * mask_expanded  # mask out padding
        scores = scores - 1e9 * (1 - mask_expanded)  # large negative for padding
        
        # Softmax attention
        attn_weights = F.softmax(scores, dim=1)  # (B, L, 1)
        attn_weights = self.attention_dropout(attn_weights)
        
        # Weighted sum pooling
        pooled = (x * attn_weights).sum(dim=1)  # (B, dim)
        
        return pooled, attn_weights.squeeze(-1)


In [6]:
class BaseModel(nn.Module):
    """Feedforward baseline model with attention pooling (fair comparison)."""
    def __init__(self, in_dim=768, hidden_dim=256, base_dim=64, attn_heads=4, dropout=0.2):
        super().__init__()
        
        # Node embedding projection
        self.node_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, base_dim),
            nn.ReLU(),
            nn.LayerNorm(base_dim)
        )
        
        # Attention pooling (same as GNN/HGNN for fair comparison)
        self.attn_pool = MultiHeadAttnPool(base_dim, n_heads=attn_heads, dropout=dropout)
        
        # Scoring head
        self.score_head = nn.Sequential(
            nn.Linear(base_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, X, nodes, mask):
        """
        X: (N, in_dim) - all node features
        nodes: (B, L) - outfit node indices
        mask: (B, L) - padding mask
        Returns: scores (B,)
        """
        # Project node features
        node_emb = self.node_proj(X)  # (N, base_dim)
        
        # Extract outfit embeddings
        emb = node_emb[nodes]  # (B, L, base_dim)
        
        # Attention pooling
        pooled, _ = self.attn_pool(emb, mask)  # (B, base_dim)
        
        # Score prediction
        scores = self.score_head(pooled).squeeze(-1)  # (B,)
        return scores

print("✅ Base Model defined (with attention pooling for fair comparison)")

✅ Base Model defined (with attention pooling for fair comparison)


## 4. Define GNN Model

In [7]:
# ============ GRAPH CONVOLUTIONAL LAYER ============
class GraphConvLayer(nn.Module):
    """Simple Graph Convolutional Layer with sparse matrix support."""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=use_bias)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, adj):
        """
        Forward pass with adjacency matrix.
        X: (N, in_dim) - node features (on GPU)
        adj: (N, N) - adjacency matrix (can be sparse, typically on CPU)
        Returns: (N, out_dim)
        """
        # Handle sparse adjacency matrix on CPU with features on GPU
        if adj.is_sparse:
            # Use sparse-dense multiplication
            X_cpu = X.cpu()
            adj_cpu = adj.cpu()  # Explicitly move to CPU
            adj_coalesced = adj_cpu.coalesce()
            
            # Perform sparse matrix multiplication on CPU
            out_cpu = torch.sparse.mm(adj_coalesced, X_cpu)  # (N, in_dim)
            
            # Move result back to original device
            out = out_cpu.to(X.device)
        else:
            # Dense case: move adj to X's device and multiply
            adj_device = adj.to(X.device)
            out = adj_device @ X  # (N, in_dim)
        
        out = self.linear(out)  # (N, out_dim)
        out = F.relu(out)
        out = self.norm(out)
        return out

# ============ SIMPLE GNN ============
class SimpleGNN(nn.Module):
    """Simple Graph Neural Network for outfit compatibility."""
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        
        # Initial projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        # Graph convolutional layers
        self.gc_layers = nn.ModuleList([
            GraphConvLayer(hidden_dim if i > 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, out_dim)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, X, adj):
        """
        X: (N, in_dim) - node features (on GPU)
        adj: (N, N) - adjacency matrix (can be sparse)
        Returns: (N, out_dim)
        """
        x = self.input_proj(X)
        x = F.relu(x)
        x = self.dropout(x)
        
        for gc_layer in self.gc_layers:
            x = gc_layer(x, adj)
            x = self.dropout(x)
        
        x = self.output_proj(x)
        x = F.relu(x)
        x = self.out_norm(x)
        x = F.normalize(x, p=2, dim=-1)
        return x

# ============ GNN OUTFIT SCORER ============
class GNNOutfitScorer(nn.Module):
    """GNN-based outfit compatibility scorer with attention pooling."""
    def __init__(self, in_dim, hidden_dim=128, gnn_dim=64, num_gnn_layers=2, 
                 attn_heads=4, dropout=0.2):
        super().__init__()
        self.gnn = SimpleGNN(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=gnn_dim,
            num_layers=num_gnn_layers,
            dropout=dropout
        )
        
        # Attention pooling
        self.attn_pool = MultiHeadAttnPool(gnn_dim, n_heads=attn_heads, dropout=dropout)
        
        # Scoring head
        self.score_head = nn.Sequential(
            nn.Linear(gnn_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, X, nodes, mask, adj):
        """
        X: (N, in_dim) - node features (on GPU)
        nodes: (B, L) - outfit node indices
        mask: (B, L) - binary mask for padding
        adj: (N, N) - adjacency matrix (can be sparse)
        Returns: scores (B,)
        """
        # Apply GNN to all nodes
        node_embeddings = self.gnn(X, adj)  # (N, gnn_dim)
        
        # Extract outfit embeddings
        emb = node_embeddings[nodes]  # (B, L, gnn_dim)
        
        # Attention pooling
        pooled, _ = self.attn_pool(emb, mask)  # (B, gnn_dim)
        
        # Score prediction
        scores = self.score_head(pooled).squeeze(-1)  # (B,)
        return scores

print("✅ GNN Model defined")

✅ GNN Model defined


## 5. Define HGNN Model

In [8]:
# ============ HYPERGRAPH CONVOLUTIONAL LAYER ============
class HypergraphConvLayer(nn.Module):
    """Hypergraph Convolutional Layer with sparse matrix support."""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=use_bias)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, H):
        """
        Hypergraph convolution using incidence matrix.
        X: (N, in_dim) - node features (on GPU)
        H: (N, E) - hypergraph incidence matrix (can be sparse, typically on CPU)
        Returns: (N, out_dim)
        """
        try:
            # Use H @ H^T to approximate neighbor aggregation
            if H.is_sparse:
                # Explicitly move to CPU before coalescing
                H_cpu = H.cpu()
                H_coalesced = H_cpu.coalesce()
                X_cpu = X.cpu()
                
                # H @ H^T on CPU - ensure both are COO format to avoid CSR beta warning
                H_t = H_coalesced.t().coalesce()  # Transpose and coalesce to COO
                HHt = torch.sparse.mm(H_coalesced, H_t)
                HHt_coo = HHt.coalesce()  # Ensure result is in COO format
                
                # Sparse-dense multiplication
                out_cpu = torch.sparse.mm(HHt_coo, X_cpu)
                out = out_cpu.to(X.device)
            else:
                HHt = H @ H.t()
                out = HHt @ X
        except Exception as e:
            # Fallback: use identity if error occurs
            out = X
        
        out = self.linear(out)
        out = F.relu(out)
        out = self.norm(out)
        return out

In [9]:
# ============ HYPERGRAPH NEURAL NETWORK ============
class HypergraphNN(nn.Module):
    """Hypergraph Neural Network for outfit compatibility."""
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        
        # Initial projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        # Hypergraph convolutional layers
        self.hgc_layers = nn.ModuleList([
            HypergraphConvLayer(hidden_dim if i > 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, out_dim)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, X, H):
        """
        X: (N, in_dim) - node features (on GPU)
        H: (N, E) - incidence matrix (can be sparse)
        Returns: (N, out_dim)
        """
        x = self.input_proj(X)
        x = F.relu(x)
        x = self.dropout(x)
        
        for hgc_layer in self.hgc_layers:
            x = hgc_layer(x, H)
            x = self.dropout(x)
        
        x = self.output_proj(x)
        x = F.relu(x)
        x = self.out_norm(x)
        x = F.normalize(x, p=2, dim=-1)
        return x

# ============ HGNN OUTFIT SCORER ============
class HGNNOutfitScorer(nn.Module):
    """HGNN-based outfit compatibility scorer with attention pooling."""
    def __init__(self, in_dim, hidden_dim=128, hgnn_dim=64, num_hgnn_layers=2, 
                 attn_heads=4, dropout=0.2):
        super().__init__()
        # Support both in_dim and input_dim parameter names for compatibility
        self.hgnn = HypergraphNN(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=hgnn_dim,
            num_layers=num_hgnn_layers,
            dropout=dropout
        )
        
        # Attention pooling
        self.attn_pool = MultiHeadAttnPool(hgnn_dim, n_heads=attn_heads, dropout=dropout)
        
        # Scoring head
        self.score_head = nn.Sequential(
            nn.Linear(hgnn_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, X, nodes, mask, H):
        """
        X: (N, in_dim) - node features (on GPU)
        nodes: (B, L) - outfit node indices
        mask: (B, L) - binary mask for padding
        H: (N, E) - incidence matrix (can be sparse)
        Returns: scores (B,)
        """
        # Apply HGNN to all nodes
        node_embeddings = self.hgnn(X, H)  # (N, hgnn_dim)
        
        # Extract outfit embeddings
        emb = node_embeddings[nodes]  # (B, L, hgnn_dim)
        
        # Attention pooling
        pooled, _ = self.attn_pool(emb, mask)  # (B, hgnn_dim)
        
        # Score prediction
        scores = self.score_head(pooled).squeeze(-1)  # (B,)
        return scores

print("✅ HGNN Model defined")

✅ HGNN Model defined


## 6. Build Graphs (Adjacency and Hypergraph)

In [10]:
print("="*70)
print("BUILDING GRAPHS")
print("="*70)

# Build adjacency matrix for training data (using remapped IDs)
print("\n📊 Building training adjacency matrix...")
edges_train = []
for outfit in train_node_ids:
    for i in range(len(outfit)):
        for j in range(i+1, len(outfit)):
            edges_train.append((outfit[i], outfit[j]))
            edges_train.append((outfit[j], outfit[i]))

row_indices = [e[0] for e in edges_train]
col_indices = [e[1] for e in edges_train]
values = [1.0] * len(edges_train)

adj_train = torch.sparse_coo_tensor(
    indices=[row_indices, col_indices],
    values=values,
    size=(N_train, N_train),
    device='cpu'  # Keep sparse tensors on CPU
)
print(f"✓ Train adjacency matrix: {adj_train.shape}, {len(edges_train)} edges")
print(f"  Node range: 0 to {N_train-1}")

# Build incidence matrix for training data (simple version)
print("\n📊 Building training incidence matrix...")
edges_list = train_node_ids  # Each outfit is a hyperedge
row_indices_H = []
col_indices_H = []

for edge_idx, edge in enumerate(edges_list):
    for node in edge:
        row_indices_H.append(node)
        col_indices_H.append(edge_idx)

H_train = torch.sparse_coo_tensor(
    indices=[row_indices_H, col_indices_H],
    values=[1.0] * len(row_indices_H),
    size=(N_train, len(edges_list)),
    device='cpu'
)
print(f"✓ Train incidence matrix: {H_train.shape}, {len(edges_list)} hyperedges")
print(f"  Node range: 0 to {N_train-1}")

# Build adjacency matrix for validation data (using remapped IDs)
print("\n📊 Building validation adjacency matrix...")
edges_val = []
for outfit in val_node_ids:
    for i in range(len(outfit)):
        for j in range(i+1, len(outfit)):
            edges_val.append((outfit[i], outfit[j]))
            edges_val.append((outfit[j], outfit[i]))

row_indices_val = [e[0] for e in edges_val]
col_indices_val = [e[1] for e in edges_val]

adj_val = torch.sparse_coo_tensor(
    indices=[row_indices_val, col_indices_val],
    values=[1.0] * len(edges_val),
    size=(N_train, N_train),  # Same size as training (same node space)
    device='cpu'
)
print(f"✓ Val adjacency matrix: {adj_val.shape}, {len(edges_val)} edges")
print(f"  Node range: 0 to {N_train-1}")

# Build validation incidence matrix
edges_list_val = val_node_ids
row_indices_H_val = []
col_indices_H_val = []

for edge_idx, edge in enumerate(edges_list_val):
    for node in edge:
        row_indices_H_val.append(node)
        col_indices_H_val.append(edge_idx)

H_val = torch.sparse_coo_tensor(
    indices=[row_indices_H_val, col_indices_H_val],
    values=[1.0] * len(row_indices_H_val),
    size=(N_train, len(edges_list_val)),  # Same node space as training
    device='cpu'
)
print(f"✓ Val incidence matrix: {H_val.shape}, {len(edges_list_val)} hyperedges")
print(f"  Node range: 0 to {N_train-1}")

# Build adjacency matrix for test data (using remapped IDs)
print("\n📊 Building test adjacency matrix...")
edges_test = []
for outfit in test_node_ids:
    for i in range(len(outfit)):
        for j in range(i+1, len(outfit)):
            edges_test.append((outfit[i], outfit[j]))
            edges_test.append((outfit[j], outfit[i]))

row_indices_test = [e[0] for e in edges_test]
col_indices_test = [e[1] for e in edges_test]

adj_test = torch.sparse_coo_tensor(
    indices=[row_indices_test, col_indices_test],
    values=[1.0] * len(edges_test),
    size=(N_test, N_test),
    device='cpu'
)
print(f"✓ Test adjacency matrix: {adj_test.shape}, {len(edges_test)} edges")
print(f"  Node range: 0 to {N_test-1}")

# Build test incidence matrix
edges_list_test = test_node_ids
row_indices_H_test = []
col_indices_H_test = []

for edge_idx, edge in enumerate(edges_list_test):
    for node in edge:
        row_indices_H_test.append(node)
        col_indices_H_test.append(edge_idx)

H_test = torch.sparse_coo_tensor(
    indices=[row_indices_H_test, col_indices_H_test],
    values=[1.0] * len(row_indices_H_test),
    size=(N_test, len(edges_list_test)),
    device='cpu'
)
print(f"✓ Test incidence matrix: {H_test.shape}, {len(edges_list_test)} hyperedges")
print(f"  Node range: 0 to {N_test-1}")

print(f"\n✅ Graphs built successfully (train, val, test)")
print(f"   Train: {adj_train.shape} adjacency, {H_train.shape} incidence")
print(f"   Val:   {adj_val.shape} adjacency, {H_val.shape} incidence")
print(f"   Test:  {adj_test.shape} adjacency, {H_test.shape} incidence")

# ============ CLEANUP: Delete node ID lists after graph building ============
del train_node_ids, val_node_ids, test_node_ids
del edges_train, edges_val, edges_test, edges_list, edges_list_val, edges_list_test
del row_indices, col_indices, row_indices_val, col_indices_val, row_indices_test, col_indices_test
del row_indices_H, col_indices_H, row_indices_H_val, col_indices_H_val, row_indices_H_test, col_indices_H_test
import gc
gc.collect()
print("\n🧹 Cleaned up node ID lists and edge indices")

BUILDING GRAPHS

📊 Building training adjacency matrix...
✓ Train adjacency matrix: torch.Size([202110, 202110]), 771568 edges
  Node range: 0 to 202109

📊 Building training incidence matrix...
✓ Train incidence matrix: torch.Size([202110, 33990]), 33990 hyperedges
  Node range: 0 to 202109

📊 Building validation adjacency matrix...
✓ Val adjacency matrix: torch.Size([202110, 202110]), 138468 edges
  Node range: 0 to 202109
✓ Val incidence matrix: torch.Size([202110, 6000]), 6000 hyperedges
  Node range: 0 to 202109

📊 Building test adjacency matrix...
✓ Test adjacency matrix: torch.Size([202110, 202110]), 661892 edges
  Node range: 0 to 202109
✓ Test incidence matrix: torch.Size([202110, 30290]), 30290 hyperedges
  Node range: 0 to 202109

✅ Graphs built successfully (train, val, test)
   Train: torch.Size([202110, 202110]) adjacency, torch.Size([202110, 33990]) incidence
   Val:   torch.Size([202110, 202110]) adjacency, torch.Size([202110, 6000]) incidence
   Test:  torch.Size([202110

## 7. Training Infrastructure

In [11]:
def train_model(model, train_loader, val_loader, adj_train=None, H_train=None, adj_val=None, H_val=None,
                epochs=20, lr=1e-3, model_name="Model", early_stopping_patience=10, early_stopping_min_delta=1e-4):
    """Generic training function with early stopping.
    
    Args:
        model: Neural network model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        adj_train: Training adjacency matrix (for GNN) - optional
        H_train: Training incidence matrix (for HGNN) - optional
        adj_val: Validation adjacency matrix (for GNN) - optional
        H_val: Validation incidence matrix (for HGNN) - optional
        epochs: Maximum number of epochs
        lr: Learning rate
        model_name: Name for printing
        early_stopping_patience: Number of epochs with no improvement to wait (default: 10)
        early_stopping_min_delta: Minimum improvement to reset patience counter (default: 1e-4)
    """
    from tqdm import tqdm
    
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    
    # Get features from datasets (train and validation separately)
    Xc_train = train_loader.dataset.Xc.to(device) if hasattr(train_loader.dataset.Xc, 'to') else torch.tensor(train_loader.dataset.Xc, dtype=torch.float32, device=device)
    Xa_train = train_loader.dataset.Xa.to(device) if hasattr(train_loader.dataset.Xa, 'to') else torch.tensor(train_loader.dataset.Xa, dtype=torch.float32, device=device)
    Xc_val = val_loader.dataset.Xc.to(device) if hasattr(val_loader.dataset.Xc, 'to') else torch.tensor(val_loader.dataset.Xc, dtype=torch.float32, device=device)
    Xa_val = val_loader.dataset.Xa.to(device) if hasattr(val_loader.dataset.Xa, 'to') else torch.tensor(val_loader.dataset.Xa, dtype=torch.float32, device=device)
    
    # Combine features for current models (Base, GNN, HGNN expect combined features)
    X_train_combined = torch.cat([Xc_train, Xa_train], dim=1)
    X_val_combined = torch.cat([Xc_val, Xa_val], dim=1)
    
    # Get number of nodes
    num_nodes_train = X_train_combined.shape[0]
    num_nodes_val = X_val_combined.shape[0]
    
    # Determine if model needs graph
    is_base = isinstance(model, BaseModel)
    is_gnn = isinstance(model, GNNOutfitScorer)
    is_hgnn = isinstance(model, HGNNOutfitScorer)
    
    best_val_loss = float('inf')
    best_state = None
    best_metrics = {}
    num_bad_epochs = 0
    
    history = {
        'train_loss': [], 'val_loss': [], 'mse': [], 'roc_auc': [],
        'accuracy': [], 'f1': []
    }
    
    print(f"\n{'='*100}")
    print(f"Training {model_name} on {device} | Epochs: {epochs} | Batches/epoch: {len(train_loader)}")
    print(f"Early Stopping: patience={early_stopping_patience}, min_delta={early_stopping_min_delta}")
    print(f"Feature shapes: Train Xc={Xc_train.shape}, Xa={Xa_train.shape}, Combined={X_train_combined.shape}")
    print(f"Feature shapes: Val   Xc={Xc_val.shape}, Xa={Xa_val.shape}, Combined={X_val_combined.shape}")
    if is_gnn and adj_train is not None:
        print(f"Using training adjacency matrix: {adj_train.shape}")
        print(f"Using validation adjacency matrix: {adj_val.shape if adj_val is not None else 'None'}")
    if is_hgnn and H_train is not None:
        print(f"Using training incidence matrix: {H_train.shape}")
        print(f"Using validation incidence matrix: {H_val.shape if H_val is not None else 'None'}")
    print(f"{'='*100}\n")
    
    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{epochs}", unit="batch", leave=False)
        
        for batch in pbar:
            optimizer.zero_grad()
            nodes = batch['nodes'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            
            # Clamp node indices to valid range [0, num_nodes_train-1]
            nodes_clamped = torch.clamp(nodes, 0, num_nodes_train - 1)
            
            # Forward pass - use training data
            if is_base:
                scores = model(X_train_combined, nodes_clamped, mask)
            elif is_gnn:
                scores = model(X_train_combined, nodes_clamped, mask, adj_train)
            elif is_hgnn:
                scores = model(X_train_combined, nodes_clamped, mask, H_train)
            else:
                scores = model(X_train_combined, nodes_clamped, mask)
            
            loss = F.binary_cross_entropy(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            pbar.set_postfix({'Loss': f'{loss.item():.6f}'})
        
        avg_train_loss = total_loss / len(train_loader.dataset)
        history['train_loss'].append(avg_train_loss)
        
        # Validation - uses validation graphs and validation features
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            for batch in val_loader:
                nodes = batch['nodes'].to(device)
                mask = batch['mask'].to(device)
                labels = batch['label'].to(device)
                
                nodes_clamped = torch.clamp(nodes, 0, num_nodes_val - 1)
                
                # Use validation graphs if available, otherwise fall back to training graphs
                if is_base:
                    scores = model(X_val_combined, nodes_clamped, mask)
                elif is_gnn:
                    val_graph = adj_val if adj_val is not None else adj_train
                    scores = model(X_val_combined, nodes_clamped, mask, val_graph)
                elif is_hgnn:
                    val_graph = H_val if H_val is not None else H_train
                    scores = model(X_val_combined, nodes_clamped, mask, val_graph)
                else:
                    scores = model(X_val_combined, nodes_clamped, mask)
                
                loss = F.binary_cross_entropy(scores, labels)
                val_loss += loss.item() * len(labels)
                
                probs = scores.cpu().numpy()
                preds = (probs > 0.5).astype(int)
                all_probs.extend(probs)
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_mse = mean_squared_error(all_labels, all_probs)
        val_f1 = f1_score(all_labels, all_preds, zero_division=0)
        val_roc_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
        val_accuracy = accuracy_score(all_labels, all_preds)
        
        history['val_loss'].append(avg_val_loss)
        history['mse'].append(val_mse)
        history['f1'].append(val_f1)
        history['roc_auc'].append(val_roc_auc)
        history['accuracy'].append(val_accuracy)
        
        # Track best val loss
        if avg_val_loss < best_val_loss - early_stopping_min_delta:
            best_val_loss = avg_val_loss
            best_metrics = {
                'val_loss': avg_val_loss,
                'mse': val_mse,
                'roc_auc': val_roc_auc,
                'accuracy': val_accuracy,
                'f1': val_f1
            }
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            num_bad_epochs = 0
        else:
            num_bad_epochs += 1
        
        # Print epoch summary (matching the requested format)
        print(f"Epoch {epoch}/{epochs} | TrainLoss={avg_train_loss:.6f} | ValLoss={avg_val_loss:.6f} | BestVal={best_val_loss:.6f} | MSE={val_mse:.6f} | ROC-AUC={val_roc_auc:.4f} | F1={val_f1:.4f}")
        
        if num_bad_epochs > early_stopping_patience:
            print(f"\nEarly stopping at epoch {epoch}. Best val_loss: {best_val_loss:.6f}")
            break
    
    # Load best model state
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n✅ Loaded best model (Val Loss: {best_val_loss:.6f})")
    
    return model, best_val_loss, best_metrics, history

print("✅ Training function defined with EARLY STOPPING (gets Xc/Xa from dataset)")

✅ Training function defined with EARLY STOPPING (gets Xc/Xa from dataset)


## 8. Train All Models

In [ ]:
print("\n" + "="*70)
print("TRAINING ALL MODELS")
print("="*70)

# Initialize models
base_model = BaseModel(in_dim=768, hidden_dim=256, dropout=0.2)
gnn_model = GNNOutfitScorer(in_dim=768, hidden_dim=128, gnn_dim=64, num_gnn_layers=2, attn_heads=4, dropout=0.2)
hgnn_model = HGNNOutfitScorer(in_dim=768, hidden_dim=128, hgnn_dim=64, num_hgnn_layers=2, attn_heads=4, dropout=0.2)

print(f"\nModel Parameters:")
print(f"  Base: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"  GNN:  {sum(p.numel() for p in gnn_model.parameters()):,}")
print(f"  HGNN: {sum(p.numel() for p in hgnn_model.parameters()):,}")

# Train models with early stopping - features extracted from dataset
print("\n🔧 Training with Xc/Xa from dataset, explicit train/val graphs...")

base_model, base_best_loss, base_best_metrics, base_history = train_model(
    base_model, train_loader, val_loader,
    adj_train=None,              # Not needed for base model
    H_train=None,                # Not needed for base model
    adj_val=None,                # Not needed for base model
    H_val=None,                  # Not needed for base model
    epochs=50, lr=1e-3, 
    model_name="Base Model",
    early_stopping_patience=10,
    early_stopping_min_delta=1e-4
)

gnn_model, gnn_best_loss, gnn_best_metrics, gnn_history = train_model(
    gnn_model, train_loader, val_loader,
    adj_train=adj_train,         # Training adjacency matrix
    H_train=None,                # Not needed for GNN
    adj_val=adj_val,             # Validation adjacency matrix
    H_val=None,                  # Not needed for GNN
    epochs=50, lr=1e-4, 
    model_name="GNN Model",
    early_stopping_patience=10,
    early_stopping_min_delta=1e-4
)

hgnn_model, hgnn_best_loss, hgnn_best_metrics, hgnn_history = train_model(
    hgnn_model, train_loader, val_loader,
    adj_train=None,              # Not needed for HGNN
    H_train=H_train,             # Training incidence matrix
    adj_val=None,                # Not needed for HGNN
    H_val=H_val,                 # Validation incidence matrix
    epochs=50, lr=1e-4, 
    model_name="HGNN Model",
    early_stopping_patience=10,
    early_stopping_min_delta=1e-4
)

print("\n" + "="*70)
print("✅ All models trained successfully with EARLY STOPPING")
print("   (Features from dataset: Xc + Xa, train/val graphs for validation)")
print("="*70)

# Store models and histories
models = {
    'Base': base_model,
    'GNN': gnn_model,
    'HGNN': hgnn_model
}

histories = {
    'Base': base_history,
    'GNN': gnn_history,
    'HGNN': hgnn_history
}

best_losses = {
    'Base': base_best_loss,
    'GNN': gnn_best_loss,
    'HGNN': hgnn_best_loss
}

best_metrics = {
    'Base': base_best_metrics,
    'GNN': gnn_best_metrics,
    'HGNN': hgnn_best_metrics
}

## 9. Test All Models

In [21]:
def evaluate_model(model, data_loader, num_nodes, model_name, adj=None, H=None, is_base=False, is_gnn=False, is_hgnn=False):
    """Evaluate model on a given dataset (features come from loader)."""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    # Build X from loader
    Xc = data_loader.dataset.Xc
    Xa = data_loader.dataset.Xa
    X = torch.cat([Xc, Xa], dim=1)
    
    with torch.no_grad():
        for batch in data_loader:
            nodes = batch['nodes'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            
            nodes = torch.clamp(nodes, 0, num_nodes - 1)
            
            if is_base:
                scores = model(X, nodes, mask)
            elif is_gnn:
                if adj is None:
                    raise ValueError("adj is required for GNN evaluation")
                scores = model(X, nodes, mask, adj.to(device))
            elif is_hgnn:
                if H is None:
                    raise ValueError("H is required for HGNN evaluation")
                scores = model(X, nodes, mask, H.to(device))
            else:
                scores = model(X, nodes, mask)
            
            probs = scores.cpu().numpy()
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'roc_auc': roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0,
        'mse': mean_squared_error(all_labels, all_probs),
    }
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    metrics['tp'] = cm[1, 1] if cm.shape == (2, 2) else 0
    metrics['fp'] = cm[0, 1] if cm.shape == (2, 2) else 0
    metrics['tn'] = cm[0, 0] if cm.shape == (2, 2) else 0
    metrics['fn'] = cm[1, 0] if cm.shape == (2, 2) else 0
    
    return metrics, all_probs, all_preds, all_labels


In [ ]:
print("\n" + "="*70)
print("TESTING ALL MODELS")
print("="*70)

test_results = {}

# Test Base Model
print("\n📊 Testing Base Model...")
base_metrics, base_probs, base_preds, base_labels = evaluate_model(
    base_model, test_loader, N_test, "Base", is_base=True
)
test_results['Base'] = {'metrics': base_metrics, 'probs': base_probs, 'preds': base_preds, 'labels': base_labels}

# Test GNN Model
print("📊 Testing GNN Model...")
gnn_metrics, gnn_probs, gnn_preds, gnn_labels = evaluate_model(
    gnn_model, test_loader, N_test, "GNN", adj=adj_test, is_gnn=True
)
test_results['GNN'] = {'metrics': gnn_metrics, 'probs': gnn_probs, 'preds': gnn_preds, 'labels': gnn_labels}

# Test HGNN Model
print("📊 Testing HGNN Model...")
hgnn_metrics, hgnn_probs, hgnn_preds, hgnn_labels = evaluate_model(
    hgnn_model, test_loader, N_test, "HGNN", H=H_test, is_hgnn=True
)
test_results['HGNN'] = {'metrics': hgnn_metrics, 'probs': hgnn_probs, 'preds': hgnn_preds, 'labels': hgnn_labels}

print("\n✅ All models tested")

# Display results
print("\n" + "="*70)
print("TEST RESULTS SUMMARY")
print("="*70)

results_df = pd.DataFrame([
    test_results['Base']['metrics'],
    test_results['GNN']['metrics'],
    test_results['HGNN']['metrics']
])

print(results_df.to_string(index=False))

# ============ CLEANUP: Delete individual metrics and unused model references ============
del base_metrics, gnn_metrics, hgnn_metrics
del base_probs, base_preds, base_labels
del gnn_probs, gnn_preds, gnn_labels
del hgnn_probs, hgnn_preds, hgnn_labels
del test_ds, test_loader, results_df
import gc
gc.collect()
print("\n🧹 Cleaned up test metrics and test dataloader")


## 9. Save Model Checkpoints

In [ ]:
# Save trained models with their metrics and histories
import json
from datetime import datetime

print("\n" + "="*80)
print("SAVING MODEL CHECKPOINTS")
print("="*80)

# Create timestamp for this training session
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create checkpoint directory structure
checkpoint_base_dir = CHECKPOINTS_PATH / "model_comparison"
checkpoint_base_dir.mkdir(parents=True, exist_ok=True)

# Dictionary to store all models
models = {
    'Base': base_model,
    'GNN': gnn_model,
    'HGNN': hgnn_model
}

histories = {
    'Base': base_history,
    'GNN': gnn_history,
    'HGNN': hgnn_history
}

best_losses = {
    'Base': base_best_loss,
    'GNN': gnn_best_loss,
    'HGNN': hgnn_best_loss
}

best_metrics = {
    'Base': base_best_metrics,
    'GNN': gnn_best_metrics,
    'HGNN': hgnn_best_metrics
}

# Save each model
for model_name in ['Base', 'GNN', 'HGNN']:
    print(f"\n{'='*80}")
    print(f"Saving {model_name} Model")
    print(f"{'='*80}")
    
    # Create experiment directory
    exp_name = f"{model_name.lower()}_model_{timestamp}"
    exp_dir = checkpoint_base_dir / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    # Store experiment directory reference
    if model_name == 'Base':
        base_exp_name = exp_name
        base_exp_dir = exp_dir
    elif model_name == 'GNN':
        gnn_exp_name = exp_name
        gnn_exp_dir = exp_dir
    else:
        hgnn_exp_name = exp_name
        hgnn_exp_dir = exp_dir
    
    # 1. Save model weights
    model_path = exp_dir / "model.pt"
    torch.save(models[model_name].state_dict(), model_path)
    print(f"  ✅ Model weights saved: {model_path.name}")
    
    # 2. Save training history
    history = histories[model_name]
    history_json = {
        'train_loss': [float(x) for x in history['train_loss']],
        'val_loss': [float(x) for x in history['val_loss']],
        'f1': [float(x) for x in history['f1']],
        'epochs': len(history['train_loss'])
    }
    
    with open(exp_dir / "history.json", 'w') as f:
        json.dump(history_json, f, indent=2)
    print(f"  ✅ Training history saved with {history_json['epochs']} epochs")
    
    # 3. Save best validation metrics
    val_metrics = best_metrics[model_name]
    metrics_json = {
        'accuracy': float(val_metrics['accuracy']),
        'precision': float(val_metrics['precision']),
        'recall': float(val_metrics['recall']),
        'f1': float(val_metrics['f1']),
        'roc_auc': float(val_metrics['roc_auc']),
        'mse': float(val_metrics['mse']),
        'best_val_loss': float(best_losses[model_name])
    }
    
    with open(exp_dir / "best_val_metrics.json", 'w') as f:
        json.dump(metrics_json, f, indent=2)
    print(f"  ✅ Best validation metrics saved")
    
    # 4. Save test metrics
    test_metrics_json = {
        'accuracy': float(test_results[model_name]['metrics']['accuracy']),
        'precision': float(test_results[model_name]['metrics']['precision']),
        'recall': float(test_results[model_name]['metrics']['recall']),
        'f1': float(test_results[model_name]['metrics']['f1']),
        'roc_auc': float(test_results[model_name]['metrics']['roc_auc']),
        'mse': float(test_results[model_name]['metrics']['mse']),
        'tp': int(test_results[model_name]['metrics']['tp']),
        'fp': int(test_results[model_name]['metrics']['fp']),
        'tn': int(test_results[model_name]['metrics']['tn']),
        'fn': int(test_results[model_name]['metrics']['fn'])
    }
    
    with open(exp_dir / "test_metrics.json", 'w') as f:
        json.dump(test_metrics_json, f, indent=2)
    print(f"  ✅ Test metrics saved")
    
    # 5. Save comprehensive summary
    summary = {
        'model_name': model_name,
        'experiment_name': exp_name,
        'timestamp': timestamp,
        'model_config': {
            'input_dim': 768,
            'type': model_name
        },
        'training_config': {
            'batch_size': batch_size,
            'max_epochs': 50,
            'patience': 10,
            'optimizer': 'AdamW'
        },
        'data_info': {
            'train_samples': len(train_loader.dataset),
            'val_samples': len(val_loader.dataset),
            'train_nodes': N_train,
            'val_nodes': N_val,
            'test_nodes': N_test
        },
        'best_validation': metrics_json,
        'test_performance': test_metrics_json,
        'training_epochs': history_json['epochs']
    }
    
    # Store summary reference for later use
    if model_name == 'Base':
        base_summary = summary
    elif model_name == 'GNN':
        gnn_summary = summary
    else:
        hgnn_summary = summary
    
    with open(exp_dir / "summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  ✅ Summary saved")
    
    print(f"\n  📁 Checkpoint saved to: {exp_dir}")
    print(f"     - model.pt")
    print(f"     - history.json")
    print(f"     - best_val_metrics.json")
    print(f"     - test_metrics.json")
    print(f"     - summary.json")

# Save combined histories for comparison
histories_combined = {
    'Base': history_json if 'Base' in models else None,
    'GNN': gnn_history,
    'HGNN': hgnn_history
}

histories_file = checkpoint_base_dir / f"all_histories_{timestamp}.json"
histories_json = {}
for name, hist in histories_combined.items():
    if hist:
        histories_json[name] = {
            'train_loss': [float(x) for x in hist['train_loss']],
            'val_loss': [float(x) for x in hist['val_loss']],
            'f1': [float(x) for x in hist['f1']]
        }

with open(histories_file, 'w') as f:
    json.dump(histories_json, f, indent=2)

print(f"\n{'='*80}")
print(f"✅ ALL MODELS SAVED SUCCESSFULLY")
print(f"{'='*80}")
print(f"\nBase directory: {checkpoint_base_dir}")
print(f"Combined histories: {histories_file.name}")
print(f"\nExperiments:")
print(f"  - Base Model: {base_exp_name}")
print(f"  - GNN Model: {gnn_exp_name}")
print(f"  - HGNN Model: {hgnn_exp_name}")

## 10. Visualize Training Curves

In [ ]:
# Plot training curves
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Training Loss', 'Validation Loss', 'Validation F1'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

colors = {'Base': '#1f77b4', 'GNN': '#ff7f0e', 'HGNN': '#2ca02c'}

for model_name, hist in histories.items():
    epochs = list(range(1, len(hist['train_loss']) + 1))
    color = colors[model_name]
    
    # Training loss
    fig.add_trace(
        go.Scatter(x=epochs, y=hist['train_loss'], mode='lines', name=f"{model_name} (train)",
                  line=dict(color=color, width=2)),
        row=1, col=1
    )
    
    # Validation loss
    fig.add_trace(
        go.Scatter(x=epochs, y=hist['val_loss'], mode='lines', name=f"{model_name} (val)",
                  line=dict(color=color, width=2, dash='dash')),
        row=1, col=2
    )
    
    # Validation F1 (key is 'f1' not 'val_f1')
    fig.add_trace(
        go.Scatter(x=epochs, y=hist['f1'], mode='lines', name=model_name,
                  line=dict(color=color, width=2), showlegend=False),
        row=1, col=3
    )

fig.update_xaxes(title_text="Epoch", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=1)
fig.update_xaxes(title_text="Epoch", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=2)
fig.update_xaxes(title_text="Epoch", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=3)
fig.update_yaxes(title_text="Loss", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=1)
fig.update_yaxes(title_text="Loss", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=2)
fig.update_yaxes(title_text="F1 Score", title_font=dict(size=14), tickfont=dict(size=12), row=1, col=3)

fig.update_layout(height=500, width=1600, title_text="Model Training Curves", font=dict(size=12), hovermode='x unified')
fig_training = fig
fig.show()

print("✅ Training curves plotted")

## 11. Compare Model Performance

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON: TRAIN / VAL / TEST")
print("="*80)

# Evaluate on training set to see overfitting
print("\n📊 Evaluating models on TRAINING SET...")
train_results = {}

for model_name in ['Base', 'GNN', 'HGNN']:
    if model_name == 'Base':
        model = base_model
    elif model_name == 'GNN':
        model = gnn_model
    else:
        model = hgnn_model
    
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    
    with torch.no_grad():
        for batch in train_loader:
            nodes = batch['nodes'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            nodes = torch.clamp(nodes, 0, N_train - 1)
            
            if model_name == 'Base':
                scores = model(X_train_combined_t, nodes, mask)
            elif model_name == 'GNN':
                scores = model(X_train_combined_t, nodes, mask, adj_train.to(device))
            else:
                scores = model(X_train_combined_t, nodes, mask, H_train.to(device))
            
            probs = scores.cpu().numpy()
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    train_metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'roc_auc': roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
    }
    train_results[model_name] = train_metrics

print("✅ Training set evaluation complete")

# Create comprehensive comparison: Train / Val / Test
print("\n\n📊 COMPLETE PERFORMANCE COMPARISON (Train / Val / Test)")
print("="*80)

comprehensive_data = []
for model_name in ['Base', 'GNN', 'HGNN']:
    # Get metrics from all sets
    train_metrics = train_results[model_name]
    
    if model_name == 'Base':
        val_metrics = base_best_metrics
    elif model_name == 'GNN':
        val_metrics = gnn_best_metrics
    else:
        val_metrics = hgnn_best_metrics
    
    test_metrics = test_results[model_name]['metrics']
    
    comprehensive_data.append({
        'Model': model_name,
        'Train F1': train_metrics['f1'],
        'Val F1': val_metrics['f1'],
        'Test F1': test_metrics['f1'],
        'Train ROC-AUC': train_metrics['roc_auc'],
        'Val ROC-AUC': val_metrics['roc_auc'],
        'Test ROC-AUC': test_metrics['roc_auc'],
        'Train Acc': train_metrics['accuracy'],
        'Val Acc': val_metrics['accuracy'],
        'Test Acc': test_metrics['accuracy']
    })

comprehensive_df = pd.DataFrame(comprehensive_data)
print(comprehensive_df.to_string(index=False))

# Test set detailed performance
print("\n\n📊 TEST SET - DETAILED METRICS")
print("="*80)
comparison_data = []
for model_name in ['Base', 'GNN', 'HGNN']:
    metrics = test_results[model_name]['metrics']
    comparison_data.append(metrics)

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Overfitting/Generalization Analysis
print("\n\n📈 OVERFITTING & GENERALIZATION ANALYSIS")
print("="*80)

for model_name in ['Base', 'GNN', 'HGNN']:
    row = comprehensive_df[comprehensive_df['Model'] == model_name].iloc[0]
    
    train_val_gap = row['Train F1'] - row['Val F1']
    val_test_gap = row['Val F1'] - row['Test F1']
    train_test_gap = row['Train F1'] - row['Test F1']
    
    print(f"\n{model_name} Model:")
    print(f"  F1 Score:")
    print(f"    Train={row['Train F1']:.4f} | Val={row['Val F1']:.4f} | Test={row['Test F1']:.4f}")
    print(f"    Train→Val gap: {train_val_gap:.4f} ({train_val_gap*100:.2f}%)")
    print(f"    Val→Test gap:  {val_test_gap:.4f} ({val_test_gap*100:.2f}%)")
    print(f"    Train→Test gap: {train_test_gap:.4f} ({train_test_gap*100:.2f}%)")
    
    print(f"  ROC-AUC:")
    print(f"    Train={row['Train ROC-AUC']:.4f} | Val={row['Val ROC-AUC']:.4f} | Test={row['Test ROC-AUC']:.4f}")
    
    # Interpret results
    if train_val_gap > 0.10:
        status = "⚠️  Strong overfitting detected"
    elif train_val_gap > 0.05:
        status = "⚠️  Moderate overfitting"
    elif train_val_gap > 0.02:
        status = "⚠️  Slight overfitting"
    else:
        status = "✅ Good fit"
    
    if abs(val_test_gap) < 0.02:
        gen_status = "✅ Excellent generalization"
    elif abs(val_test_gap) < 0.05:
        gen_status = "✓ Good generalization"
    else:
        gen_status = "⚠️  Poor generalization"
    
    print(f"  Overfitting: {status}")
    print(f"  Generalization: {gen_status}")

# Visualization: Metrics Comparison
fig_metrics = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'MSE'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

colors_models = ['#1f77b4', '#ff7f0e', '#2ca02c']
model_names = ['Base', 'GNN', 'HGNN']

metrics_to_plot = [
    ('accuracy', 'Accuracy', 1, 1),
    ('precision', 'Precision', 1, 2),
    ('recall', 'Recall', 1, 3),
    ('f1', 'F1 Score', 2, 1),
    ('roc_auc', 'ROC-AUC', 2, 2),
    ('mse', 'MSE', 2, 3)
]

for metric_key, metric_name, row, col in metrics_to_plot:
    values = [test_results[m]['metrics'][metric_key] for m in model_names]
    
    fig_metrics.add_trace(
        go.Bar(
            x=model_names,
            y=values,
            marker=dict(color=colors_models),
            text=[f'{v:.4f}' for v in values],
            textposition='outside',
            textfont=dict(size=13, color='black', family='Arial'),
            showlegend=False,
            name=metric_name,
            hovertemplate='<b>%{x}</b><br>%{y:.4f}<extra></extra>'
        ),
        row=row, col=col
    )

# Update all axes to have consistent font sizes
for i in range(1, 3):
    for j in range(1, 4):
        fig_metrics.update_xaxes(tickfont=dict(size=11), row=i, col=j)
        fig_metrics.update_yaxes(tickfont=dict(size=11), row=i, col=j)

fig_metrics.update_layout(
    height=1100, 
    width=1800, 
    title_text="Model Comparison: All Metrics", 
    title_font=dict(size=18, color='#333333'),
    font=dict(size=13, family='Arial'),
    margin=dict(t=120, b=120, l=100, r=100),
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    paper_bgcolor='white',
    showlegend=False,
    hovermode='closest'
)
fig_metrics.show()

print("\n✅ Comparison visualizations created")

# ROC Curves
fig_roc = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'{m} Model' for m in model_names],
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

for idx, model_name in enumerate(model_names, 1):
    labels = test_results[model_name]['labels']
    probs = test_results[model_name]['probs']
    
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    
    fig_roc.add_trace(
        go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC-AUC = {roc_auc:.4f}',
                  line=dict(color=colors_models[idx-1], width=2.5),
                  fill='tozeroy', fillcolor=colors_models[idx-1], opacity=0.3),
        row=1, col=idx
    )
    
    fig_roc.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(color='black', dash='dash'),
                  showlegend=False, hoverinfo='skip'),
        row=1, col=idx
    )

fig_roc.update_xaxes(title_text="FPR", row=1, col=2)
fig_roc.update_yaxes(title_text="TPR", row=1, col=1)
fig_roc.update_layout(height=450, width=1200, title_text="ROC Curves Comparison")
fig_roc.show()

print("✅ ROC curves plotted")

# Confusion Matrices
fig_cm = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'{m} Model' for m in model_names],
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}, {'type': 'heatmap'}]]
)

for idx, model_name in enumerate(model_names, 1):
    labels = test_results[model_name]['labels']
    preds = test_results[model_name]['preds']
    
    cm = confusion_matrix(labels, preds)
    
    fig_cm.add_trace(
        go.Heatmap(z=cm, x=['Neg', 'Pos'], y=['Neg', 'Pos'],
                  colorscale='Blues', text=cm, texttemplate='%{text}',
                  showscale=(idx==3), colorbar=dict(len=0.5, y=0.5) if idx==3 else None),
        row=1, col=idx
    )

fig_cm.update_yaxes(title_text="True", row=1, col=1)
fig_cm.update_xaxes(title_text="Predicted", row=1, col=3)
fig_cm.update_layout(height=400, width=1200, title_text="Confusion Matrices Comparison")
fig_cm.show()

print("✅ Confusion matrices plotted")

### Train/Val/Test Performance Comparison Visualizations

Visual comparison of how each model performs across training, validation, and test datasets.

In [ ]:
# Grouped Bar Chart: Train/Val/Test comparison
print("\n" + "="*80)
print("CREATING TRAIN/VAL/TEST VISUALIZATION CHARTS")
print("="*80)

fig_comparison = make_subplots(
    rows=1, cols=3,
    subplot_titles=('F1 Score Across Datasets', 'ROC-AUC Across Datasets', 'Accuracy Across Datasets'),
    horizontal_spacing=0.1
)

colors_map = {'Base': '#1f77b4', 'GNN': '#ff7f0e', 'HGNN': '#2ca02c'}
model_names_viz = ['Base', 'GNN', 'HGNN']
datasets = ['Train', 'Validation', 'Test']

# Get validation metrics properly
val_metrics_map = {
    'Base': base_best_metrics,
    'GNN': gnn_best_metrics,
    'HGNN': hgnn_best_metrics
}

plot_data = {
    'F1 Score': {},
    'ROC-AUC': {},
    'Accuracy': {}
}

for model in model_names_viz:
    train_m = train_results[model]
    val_m = val_metrics_map[model]
    test_m = test_results[model]['metrics']
    
    plot_data['F1 Score'][model] = [train_m['f1'], val_m['f1'], test_m['f1']]
    plot_data['ROC-AUC'][model] = [train_m['roc_auc'], val_m['roc_auc'], test_m['roc_auc']]
    plot_data['Accuracy'][model] = [train_m['accuracy'], val_m['accuracy'], test_m['accuracy']]

# Add traces for each metric
for col_idx, (metric_name, metric_data) in enumerate(plot_data.items(), start=1):
    for model in model_names_viz:
        fig_comparison.add_trace(
            go.Bar(
                name=model,
                x=datasets,
                y=metric_data[model],
                text=[f'{v:.4f}' for v in metric_data[model]],
                textposition='outside',
                marker_color=colors_map[model],
                showlegend=(col_idx == 1),  # Only show legend for first subplot
                legendgroup=model
            ),
            row=1, col=col_idx
        )

fig_comparison.update_layout(
    height=600,
    width=1800,
    title_text="<b>Model Performance Comparison: Train vs Validation vs Test</b>",
    title_font_size=16,
    barmode='group',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    )
)

# Update all y-axes
for i in range(1, 4):
    fig_comparison.update_yaxes(title_text="Score", range=[0, 1.05], row=1, col=i)
    fig_comparison.update_xaxes(title_text="Dataset", row=1, col=i)

fig_comparison.show()

# Save the figure
comparison_viz_html = comparison_dir / f'train_val_test_comparison_{timestamp}.html'
comparison_viz_png = comparison_dir / f'train_val_test_comparison_{timestamp}.png'
fig_comparison.write_html(str(comparison_viz_html))
fig_comparison.write_image(str(comparison_viz_png))

print(f"\n✅ Train/Val/Test comparison visualization saved:")
print(f"   - HTML: {comparison_viz_html}")
print(f"   - PNG: {comparison_viz_png}")

In [ ]:
# Line Plot: F1 Score Progression
fig_progression = go.Figure()

for model in model_names_viz:
    train_m = train_results[model]
    val_m = val_metrics_map[model]
    test_m = test_results[model]['metrics']
    
    # F1 Score progression
    fig_progression.add_trace(go.Scatter(
        x=datasets,
        y=[train_m['f1'], val_m['f1'], test_m['f1']],
        mode='lines+markers+text',
        name=f'{model} (F1)',
        line=dict(width=3, color=colors_map[model]),
        marker=dict(size=12, symbol='circle'),
        text=[f"{v:.4f}" for v in [train_m['f1'], val_m['f1'], test_m['f1']]],
        textposition="top center",
        textfont=dict(size=11)
    ))

fig_progression.update_layout(
    title="<b>F1 Score Progression: Train → Validation → Test</b>",
    title_font_size=16,
    xaxis_title="Dataset",
    yaxis_title="F1 Score",
    height=600,
    width=1200,
    template='plotly_white',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        font=dict(size=12)
    ),
    yaxis=dict(range=[0.5, 1.0], gridcolor='lightgray'),
    xaxis=dict(gridcolor='lightgray'),
    hovermode='x unified'
)

fig_progression.show()

# Save the progression figure
progression_html = comparison_dir / f'f1_progression_{timestamp}.html'
progression_png = comparison_dir / f'f1_progression_{timestamp}.png'
fig_progression.write_html(str(progression_html))
fig_progression.write_image(str(progression_png))

print(f"\n✅ F1 progression visualization saved:")
print(f"   - HTML: {progression_html}")
print(f"   - PNG: {progression_png}")

In [ ]:
# Performance Gaps Analysis
fig_gaps = go.Figure()

gap_types = ['Train→Val Gap', 'Val→Test Gap', 'Train→Test Gap']

for model in model_names_viz:
    train_m = train_results[model]
    val_m = val_metrics_map[model]
    test_m = test_results[model]['metrics']
    
    # Calculate gaps (positive gap = performance drop)
    train_val_gap = (train_m['f1'] - val_m['f1']) * 100
    val_test_gap = (val_m['f1'] - test_m['f1']) * 100
    train_test_gap = (train_m['f1'] - test_m['f1']) * 100
    
    gaps = [train_val_gap, val_test_gap, train_test_gap]
    
    fig_gaps.add_trace(go.Bar(
        name=model,
        x=gap_types,
        y=gaps,
        text=[f'{g:+.2f}%' for g in gaps],
        textposition='outside',
        marker_color=colors_map[model]
    ))

fig_gaps.update_layout(
    title="<b>Performance Gaps Analysis (F1 Score)</b><br><sub>Positive values indicate performance drop</sub>",
    title_font_size=16,
    xaxis_title="Gap Type",
    yaxis_title="Performance Drop (%)",
    height=600,
    width=1200,
    template='plotly_white',
    barmode='group',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    ),
    yaxis=dict(gridcolor='lightgray', zeroline=True, zerolinewidth=2, zerolinecolor='black'),
    xaxis=dict(gridcolor='lightgray')
)

fig_gaps.show()

# Save the gaps figure
gaps_html = comparison_dir / f'performance_gaps_{timestamp}.html'
gaps_png = comparison_dir / f'performance_gaps_{timestamp}.png'
fig_gaps.write_html(str(gaps_html))
fig_gaps.write_image(str(gaps_png))

print(f"\n✅ Performance gaps visualization saved:")
print(f"   - HTML: {gaps_html}")
print(f"   - PNG: {gaps_png}")

In [ ]:
# Summary Report
print("\n" + "="*80)
print("FINAL COMPARISON SUMMARY")
print("="*80)

# Determine winner
metrics_columns = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
scores = {'Base': 0, 'GNN': 0, 'HGNN': 0}

for model_name in model_names:
    for metric in metrics_columns:
        val = test_results[model_name]['metrics'][metric]
        if val == max(test_results[m]['metrics'][metric] for m in model_names):
            scores[model_name] += 1

print("\nMetric Wins:")
for model_name, wins in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model_name:10s}: {wins} wins")

best_model = max(scores, key=scores.get)
print(f"\n🏆 Best Model: {best_model}")
print(f"   Accuracy: {test_results[best_model]['metrics']['accuracy']:.4f}")
print(f"   F1 Score: {test_results[best_model]['metrics']['f1']:.4f}")
print(f"   ROC-AUC: {test_results[best_model]['metrics']['roc_auc']:.4f}")

print("\n" + "="*80)

# ============ FINAL CLEANUP ============
print("\n🧹 FINAL MEMORY CLEANUP")
print("="*80)

# Keep only what's needed: models and test_results
# Delete histories and other temporary variables
import gc
gc.collect()

print("\n✅ Memory optimization complete!")
print("   Retained: models dict, test_results dict, best_model reference")
print("   Freed: DataFrames, dataloaders, intermediate arrays, histories")

# Optional: Get memory info
import psutil
import os
process = psutil.Process(os.getpid())
mem_info = process.memory_info()
print(f"\n📊 Current memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

In [ ]:

# Save Comparison Figures and Metrics
print("\n" + "="*80)
print("SAVING COMPARISON RESULTS")
print("="*80)

# Create results directory
comparison_dir = checkpoint_base_dir / "comparison_results"
comparison_dir.mkdir(parents=True, exist_ok=True)

# ============ SAVE COMPARISON METRICS CSV ============
print("\n📊 Saving comparison metrics...")

# Create comprehensive comparison CSV with Train/Val/Test
comprehensive_comparison_data = []
for model_name in ['Base', 'GNN', 'HGNN']:
    # Get all metrics
    train_metrics = train_results[model_name]
    
    if model_name == 'Base':
        val_metrics = base_best_metrics
    elif model_name == 'GNN':
        val_metrics = gnn_best_metrics
    else:
        val_metrics = hgnn_best_metrics
    
    test_metrics = test_results[model_name]['metrics']
    
    # Calculate gaps
    train_val_gap_f1 = train_metrics['f1'] - val_metrics['f1']
    val_test_gap_f1 = val_metrics['f1'] - test_metrics['f1']
    train_test_gap_f1 = train_metrics['f1'] - test_metrics['f1']
    
    comprehensive_comparison_data.append({
        'model': model_name,
        # Train metrics
        'train_f1': train_metrics['f1'],
        'train_roc_auc': train_metrics['roc_auc'],
        'train_accuracy': train_metrics['accuracy'],
        # Validation metrics (use .get() with defaults for metrics that may not exist)
        'val_f1': val_metrics['f1'],
        'val_roc_auc': val_metrics['roc_auc'],
        'val_accuracy': val_metrics['accuracy'],
        'val_precision': val_metrics.get('precision', test_metrics['precision']),
        'val_recall': val_metrics.get('recall', test_metrics['recall']),
        'val_mse': val_metrics.get('mse', test_metrics['mse']),
        # Test metrics
        'test_f1': test_metrics['f1'],
        'test_roc_auc': test_metrics['roc_auc'],
        'test_accuracy': test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_mse': test_metrics['mse'],
        'test_tp': test_metrics['tp'],
        'test_fp': test_metrics['fp'],
        'test_tn': test_metrics['tn'],
        'test_fn': test_metrics['fn'],
        # Performance gaps
        'train_val_gap_f1': train_val_gap_f1,
        'val_test_gap_f1': val_test_gap_f1,
        'train_test_gap_f1': train_test_gap_f1
    })

comprehensive_comparison_df = pd.DataFrame(comprehensive_comparison_data)
comparison_csv = comparison_dir / f"model_comparison_train_val_test_{timestamp}.csv"
comprehensive_comparison_df.to_csv(comparison_csv, index=False)
print(f"✓ Comprehensive metrics saved: {comparison_csv.name}")

# Also save just test metrics for backward compatibility
test_only_csv = comparison_dir / f"model_comparison_test_only_{timestamp}.csv"
comparison_df.to_csv(test_only_csv, index=False)
print(f"✓ Test-only metrics saved: {test_only_csv.name}")

# ============ SAVE FIGURES AS HTML ============
print("\n📈 Saving interactive figures as HTML...")

training_curves_html = comparison_dir / f"01_training_curves_{timestamp}.html"
fig_training.write_html(str(training_curves_html))
print(f"✓ Training curves: {training_curves_html.name}")

metrics_html = comparison_dir / f"02_metrics_comparison_{timestamp}.html"
fig_metrics.write_html(str(metrics_html))
print(f"✓ Metrics comparison: {metrics_html.name}")

roc_html = comparison_dir / f"03_roc_curves_{timestamp}.html"
fig_roc.write_html(str(roc_html))
print(f"✓ ROC curves: {roc_html.name}")

cm_html = comparison_dir / f"04_confusion_matrices_{timestamp}.html"
fig_cm.write_html(str(cm_html))
print(f"✓ Confusion matrices: {cm_html.name}")

# ============ SAVE FIGURES AS PNG ============
print("\n🖼️  Saving figures as PNG...")

try:
    training_curves_png = comparison_dir / f"01_training_curves_{timestamp}.png"
    fig_training.write_image(str(training_curves_png), width=1600, height=500)
    print(f"✓ Training curves PNG: {training_curves_png.name}")
except Exception as e:
    print(f"  ⚠️  PNG save skipped (kaleido not available): {e}")

try:
    metrics_png = comparison_dir / f"02_metrics_comparison_{timestamp}.png"
    fig_metrics.write_image(str(metrics_png), width=1600, height=1100)
    print(f"✓ Metrics comparison PNG: {metrics_png.name}")
except Exception as e:
    print(f"  ⚠️  PNG save skipped (kaleido not available): {e}")

try:
    roc_png = comparison_dir / f"03_roc_curves_{timestamp}.png"
    fig_roc.write_image(str(roc_png), width=1200, height=450)
    print(f"✓ ROC curves PNG: {roc_png.name}")
except Exception as e:
    print(f"  ⚠️  PNG save skipped (kaleido not available): {e}")

try:
    cm_png = comparison_dir / f"04_confusion_matrices_{timestamp}.png"
    fig_cm.write_image(str(cm_png), width=1200, height=400)
    print(f"✓ Confusion matrices PNG: {cm_png.name}")
except Exception as e:
    print(f"  ⚠️  PNG save skipped (kaleido not available): {e}")

# ============ FINAL SUMMARY ============
print("\n" + "="*80)
print("✅ COMPARISON RESULTS SAVED")
print("="*80)
print(f"\nResults Directory: {comparison_dir}")
print(f"\nFiles Generated:")
print(f"  📊 CSV Files:")
print(f"     - model_comparison_train_val_test_{timestamp}.csv (comprehensive)")
print(f"     - model_comparison_test_only_{timestamp}.csv (test metrics only)")
print(f"  📈 Figures (HTML): 01_training_curves, 02_metrics_comparison, 03_roc_curves, 04_confusion_matrices")
print(f"  🖼️  Figures (PNG): 01_training_curves, 02_metrics_comparison, 03_roc_curves, 04_confusion_matrices")
print("="*80)


# 12. HGNN Hyperparameter Search

Grid search to find optimal HGNN hyperparameters

In [ ]:
# Define hyperparameter search using RANDOM SEARCH (more efficient than grid search)
print("\n" + "="*80)
print("HGNN HYPERPARAMETER SEARCH (RANDOM SEARCH)")
print("="*80)

import numpy as np
import random
from scipy.stats import loguniform

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

# Define hyperparameter distributions - EXPANDED to include more HGNN-specific parameters
# Using tighter log-uniform ranges so LR decays are stable (fast but not too fast)
hyperparameter_distributions = {
    # Architecture parameters
    'hidden_dim': [64, 128, 256, 512],          # Main model hidden dimension
    'hgnn_dim': [32, 64, 128, 256],              # HGNN output dimension
    'num_hgnn_layers': [1, 2, 3, 4],             # Number of HGNN layers
    'attn_heads': [1, 2, 4, 8],                  # Attention heads in pooling
    # Training parameters (log-uniform for better scaling)
    'learning_rate': loguniform(1e-3, 1e-2),    # Moderate range to converge quickly without overshooting
    'weight_decay': loguniform(1e-6, 3e-4),     # Smaller decay to avoid underfitting from over-regularization
    'dropout': [0.1, 0.2, 0.3, 0.4, 0.5]        # Dropout rate
}

# Number of random trials (better exploration than grid with fewer experiments)
n_trials = 50  # Increased from 20 to cover expanded parameter space

# Generate random hyperparameter combinations
param_combinations = []
for trial_idx in range(n_trials):
    config = {
        'hidden_dim': random.choice(hyperparameter_distributions['hidden_dim']),
        'hgnn_dim': random.choice(hyperparameter_distributions['hgnn_dim']),
        'num_hgnn_layers': random.choice(hyperparameter_distributions['num_hgnn_layers']),
        'attn_heads': random.choice(hyperparameter_distributions['attn_heads']),
        'learning_rate': hyperparameter_distributions['learning_rate'].rvs(),
        'weight_decay': hyperparameter_distributions['weight_decay'].rvs(),
        'dropout': random.choice(hyperparameter_distributions['dropout']),
        'trial': trial_idx + 1
    }
    config['name'] = (f"hgnn_trial{trial_idx+1:02d}_h{config['hidden_dim']}_hd{config['hgnn_dim']}_"
                      f"nl{config['num_hgnn_layers']}_ah{config['attn_heads']}_"
                      f"lr{config['learning_rate']:.2e}_wd{config['weight_decay']:.2e}_"
                      f"dr{int(config['dropout']*100)}")
    param_combinations.append(config)

# Sort by trial number for easier tracking
param_combinations = sorted(param_combinations, key=lambda x: x['trial'])

print(f"\n📊 Expanded Random Search Configuration:")
print(f"   - Total parameters: 7 (hidden_dim, hgnn_dim, num_hgnn_layers, attn_heads, lr, wd, dropout)")
print(f"   - Total trials: {len(param_combinations)}")
print(f"   - Estimated time: ~{len(param_combinations) * 5} minutes (assuming 5 min/trial)")
print(f"   - Method: Random sampling from 7-dimensional parameter space")
print(f"   - Advantage: Explores HGNN architecture + training hyperparameters simultaneously")

# Show parameter space statistics
print(f"\n📐 Parameter Space:")
for param_name, values in [('hidden_dim', [64, 128, 256, 512]), 
                            ('hgnn_dim', [32, 64, 128, 256]),
                            ('num_hgnn_layers', [1, 2, 3, 4]),
                            ('attn_heads', [1, 2, 4, 8]),
                            ('learning_rate', 'log-uniform [1e-4, 1e-2]'),
                            ('weight_decay', 'log-uniform [1e-5, 1e-2]'),
                            ('dropout', [0.1, 0.2, 0.3, 0.4, 0.5])]:
    if isinstance(values, list):
        print(f"   - {param_name:20s}: {len(values)} options {values}")
    else:
        print(f"   - {param_name:20s}: {values}")

# Show sample configurations
print("\n🔍 Sample configurations (first 5 trials):")
for config in param_combinations[:5]:
    print(f"  Trial {config['trial']:02d}: {config['name']}")
    print(f"     - Architecture: hidden={config['hidden_dim']}, hgnn_dim={config['hgnn_dim']}, "
          f"layers={config['num_hgnn_layers']}, attn_heads={config['attn_heads']}")
    print(f"     - Training: lr={config['learning_rate']:.2e}, wd={config['weight_decay']:.2e}, "
          f"dropout={config['dropout']}")

print(f"\n... and {len(param_combinations) - 5} more random trials")

In [ ]:
# Run hyperparameter search
import time
from datetime import datetime

# Storage for all experiment results
all_experiments = {}
hparam_search_dir = checkpoint_base_dir / "hgnn_hyperparam_search"
hparam_search_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "="*80)
print("RUNNING EXPERIMENTS")
print("="*80)

start_time = time.time()

for exp_idx, params in enumerate(param_combinations, 1):
    exp_name = params['name']
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {exp_idx}/{len(param_combinations)}: {exp_name}")
    print(f"{'='*80}")
    print(f"Architecture: hidden={params['hidden_dim']}, hgnn_dim={params['hgnn_dim']}, "
          f"layers={params['num_hgnn_layers']}, attn_heads={params['attn_heads']}")
    print(f"Training: lr={params['learning_rate']:.2e}, wd={params['weight_decay']:.2e}, dropout={params['dropout']}")
    
    exp_start = time.time()
    
    # Create model with current hyperparameters
    model = HGNNOutfitScorer(
        in_dim=X_train_combined_t.shape[1],
        hidden_dim=params['hidden_dim'],
        hgnn_dim=params['hgnn_dim'],
        num_hgnn_layers=params['num_hgnn_layers'],
        attn_heads=params['attn_heads'],
        dropout=params['dropout']
    )
    
    # Train the model using the standard train_model function
    model, best_val_loss, best_metrics, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        adj_train=None,
        H_train=H_train,
        adj_val=None,
        H_val=H_val,
        epochs=25,
        lr=params['learning_rate'],
        model_name=exp_name,
        early_stopping_patience=5,
        early_stopping_min_delta=1e-4
    )
    
    # Evaluate on test set using evaluate_model function
    test_metrics, _, _, _ = evaluate_model(
        model, test_loader, X_test_combined_t, N_test, exp_name, is_hgnn=True
    )
    
    # Get best validation metrics from history
    best_val_metrics = {
        'val_loss': best_val_loss,
        'val_f1': best_metrics.get('f1', max(history['f1']) if history['f1'] else 0.0),
        'val_roc_auc': best_metrics.get('roc_auc', 0.0)
    }
    
    exp_duration = time.time() - exp_start
    
    # Store results
    all_experiments[exp_name] = {
        'params': params,
        'history': history,
        'test_metrics': test_metrics,
        'best_val_metrics': best_val_metrics,
        'duration_seconds': exp_duration
    }
    
    # Save experiment checkpoint
    exp_dir = hparam_search_dir / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    # Convert dictionaries to native Python types for JSON serialization
    def to_serializable(obj):
        if isinstance(obj, (np.integer, int)):
            return int(obj)
        if isinstance(obj, (np.floating, float)):
            return float(obj)
        if isinstance(obj, dict):
            return {k: to_serializable(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [to_serializable(v) for v in obj]
        return obj
    
    # 1. Save model weights
    torch.save(model.state_dict(), exp_dir / "model.pt")
    print(f"  ✅ Model weights saved")
    
    # 2. Save training history
    history_json = {
        'train_loss': [float(x) for x in history['train_loss']],
        'val_loss': [float(x) for x in history['val_loss']],
        'mse': [float(x) for x in history['mse']],
        'roc_auc': [float(x) for x in history['roc_auc']],
        'accuracy': [float(x) for x in history['accuracy']],
        'f1': [float(x) for x in history['f1']],
        'epochs': len(history['train_loss'])
    }
    with open(exp_dir / "history.json", 'w') as f:
        json.dump(history_json, f, indent=2)
    print(f"  ✅ Training history saved ({history_json['epochs']} epochs)")
    
    # 3. Save best validation metrics
    best_val_metrics_json = to_serializable(best_val_metrics)
    best_val_metrics_json['val_loss'] = float(best_val_loss)
    best_val_metrics_json['accuracy'] = float(best_metrics.get('accuracy', 0.0))
    best_val_metrics_json['precision'] = float(best_metrics.get('precision', 0.0))
    best_val_metrics_json['recall'] = float(best_metrics.get('recall', 0.0))
    best_val_metrics_json['mse'] = float(best_metrics.get('mse', 0.0))
    
    with open(exp_dir / "best_val_metrics.json", 'w') as f:
        json.dump(best_val_metrics_json, f, indent=2)
    print(f"  ✅ Best validation metrics saved")
    
    # 4. Save test metrics
    test_metrics_json = to_serializable(test_metrics)
    with open(exp_dir / "test_metrics.json", 'w') as f:
        json.dump(test_metrics_json, f, indent=2)
    print(f"  ✅ Test metrics saved")
    
    # 5. Save comprehensive summary
    summary = {
        'model_name': 'HGNN',
        'experiment_name': exp_name,
        'trial_number': params.get('trial', exp_idx),
        'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
        'hyperparameters': to_serializable(params),
        'model_config': {
            'input_dim': X_train_combined_t.shape[1],
            'hidden_dim': params['hidden_dim'],
            'hgnn_dim': params['hgnn_dim'],
            'num_hgnn_layers': params['num_hgnn_layers'],
            'attn_heads': params['attn_heads'],
            'dropout': params['dropout']
        },
        'training_config': {
            'batch_size': batch_size,
            'max_epochs': 25,
            'learning_rate': float(params['learning_rate']),
            'weight_decay': float(params['weight_decay']),
            'patience': 5,
            'min_delta': 1e-4,
            'optimizer': 'AdamW'
        },
        'data_info': {
            'train_samples': len(train_loader.dataset),
            'val_samples': len(val_loader.dataset),
            'train_nodes': N_train,
            'val_nodes': N_val,
            'test_nodes': N_test
        },
        'best_validation': best_val_metrics_json,
        'test_performance': test_metrics_json,
        'training_epochs': history_json['epochs'],
        'duration_seconds': float(exp_duration)
    }
    
    with open(exp_dir / "summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  ✅ Summary saved")
    
    print(f"\n  📁 Checkpoint: {exp_dir.name}")
    print(f"     - model.pt")
    print(f"     - history.json")
    print(f"     - best_val_metrics.json")
    print(f"     - test_metrics.json")
    print(f"     - summary.json")
    
    print(f"\n📊 Trial {params.get('trial', exp_idx)} Results:")
    print(f"   Val Loss: {best_val_metrics['val_loss']:.4f}")
    print(f"   Val F1: {best_val_metrics['val_f1']:.4f}")
    print(f"   Test F1: {test_metrics['f1']:.4f}")
    print(f"   Test ROC-AUC: {test_metrics['roc_auc']:.4f}")
    print(f"   ⏱️  Duration: {exp_duration:.1f}s")
    
    # Memory cleanup
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

total_duration = time.time() - start_time

print(f"\n{'='*80}")
print(f"✅ ALL EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total duration: {total_duration/60:.1f} minutes")
print(f"Average per experiment: {total_duration/len(param_combinations):.1f} seconds")

In [ ]:
# Analyze results and find best hyperparameters with statistical insights
print("\n" + "="*80)
print("RANDOM SEARCH RESULTS & ANALYSIS")
print("="*80)

# Create comparison dataframe
hparam_results = []
for exp_name, data in all_experiments.items():
    params = data['params']
    test_metrics = data['test_metrics']
    val_metrics = data['best_val_metrics']
    
    hparam_results.append({
        'trial': params.get('trial', 0),
        'experiment': exp_name,
        'hidden_dim': params['hidden_dim'],
        'hgnn_dim': params['hgnn_dim'],
        'num_hgnn_layers': params['num_hgnn_layers'],
        'attn_heads': params['attn_heads'],
        'learning_rate': params['learning_rate'],
        'weight_decay': params['weight_decay'],
        'dropout': params['dropout'],
        'val_f1': val_metrics['val_f1'],
        'val_loss': val_metrics['val_loss'],
        'test_f1': test_metrics['f1'],
        'test_roc_auc': test_metrics['roc_auc'],
        'test_accuracy': test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'duration_min': data['duration_seconds'] / 60
    })

hparam_df = pd.DataFrame(hparam_results)

# Sort by validation F1 (descending)
hparam_df_sorted = hparam_df.sort_values('val_f1', ascending=False).reset_index(drop=True)

# Display statistics
print("\n📊 SEARCH STATISTICS:")
print(f"   Total Trials: {len(hparam_df)}")
print(f"   Avg Val F1: {hparam_df['val_f1'].mean():.4f} ± {hparam_df['val_f1'].std():.4f}")
print(f"   Avg Test F1: {hparam_df['test_f1'].mean():.4f} ± {hparam_df['test_f1'].std():.4f}")
print(f"   Best Val F1: {hparam_df['val_f1'].max():.4f}")
print(f"   Best Test F1: {hparam_df['test_f1'].max():.4f}")
print(f"   Total Time: {hparam_df['duration_min'].sum():.1f} minutes")

# Display top 10 configurations
print("\n🏆 TOP 10 CONFIGURATIONS (by Validation F1):")
print("="*80)
top_10_display = hparam_df_sorted.head(10)[['trial', 'hidden_dim', 'hgnn_dim', 'num_hgnn_layers',
                                             'attn_heads', 'learning_rate', 'weight_decay',
                                             'dropout', 'val_f1', 'test_f1', 'test_roc_auc']]
print(top_10_display.to_string(index=False))

# Best configuration
best_config = hparam_df_sorted.iloc[0]
print(f"\n{'='*80}")
print(f"✨ BEST CONFIGURATION (Trial {best_config['trial']}):")
print(f"{'='*80}")
print(f"Experiment: {best_config['experiment']}")
print(f"\nArchitecture:")
print(f"  - Hidden Dimension: {best_config['hidden_dim']}")
print(f"  - HGNN Output Dimension: {best_config['hgnn_dim']}")
print(f"  - HGNN Layers: {best_config['num_hgnn_layers']}")
print(f"  - Attention Heads: {best_config['attn_heads']}")
print(f"  - Dropout: {best_config['dropout']}")
print(f"\nTraining Parameters:")
print(f"  - Learning Rate: {best_config['learning_rate']:.4e}")
print(f"  - Weight Decay: {best_config['weight_decay']:.4e}")
print(f"\nValidation Metrics:")
print(f"  - F1: {best_config['val_f1']:.4f}")
print(f"  - Loss: {best_config['val_loss']:.4f}")
print(f"\nTest Metrics:")
print(f"  - F1: {best_config['test_f1']:.4f}")
print(f"  - ROC-AUC: {best_config['test_roc_auc']:.4f}")
print(f"  - Accuracy: {best_config['test_accuracy']:.4f}")
print(f"  - Precision: {best_config['test_precision']:.4f}")
print(f"  - Recall: {best_config['test_recall']:.4f}")
print(f"\nTraining Duration: {best_config['duration_min']:.2f} minutes")

# Save results
hparam_csv = hparam_search_dir / "hyperparameter_search_results.csv"
hparam_df_sorted.to_csv(hparam_csv, index=False)
print(f"\n✅ Full results saved to: {hparam_csv}")

In [ ]:
# Create random search performance distribution visualization
print("\n📈 Creating random search analysis visualizations...")

# 1. Learning curves showing trial convergence
fig_trials = go.Figure()

# Plot test F1 scores across trials
fig_trials.add_trace(go.Scatter(
    x=hparam_df_sorted['trial'],
    y=hparam_df_sorted['test_f1'],
    mode='markers+lines',
    marker=dict(size=8, color=hparam_df_sorted['test_f1'], colorscale='Viridis'),
    name='Test F1',
    showlegend=True
))

# Add best score line
best_test_f1 = hparam_df_sorted['test_f1'].max()
fig_trials.add_hline(y=best_test_f1, line_dash="dash", line_color="red", 
                    annotation_text=f"Best: {best_test_f1:.4f}")

fig_trials.update_layout(
    title="Random Search: Trial Performance (Test F1 Score)",
    xaxis_title="Trial Number",
    yaxis_title="Test F1 Score",
    height=500,
    width=1200,
    template='plotly_white',
    hovermode='x unified'
)
fig_trials.show()

# 2. Parameter importance scatter plots - EXPANDED to include HGNN-specific parameters
fig_params = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Hidden Dim vs Test F1', 'HGNN Dim vs Test F1', 'Num Layers vs Test F1',
                   'Attn Heads vs Test F1', 'Learning Rate vs Test F1', 'Dropout vs Test F1'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

# Hidden Dimension
fig_params.add_trace(
    go.Scatter(x=hparam_df['hidden_dim'], y=hparam_df['test_f1'], 
              mode='markers', name='Hidden Dim', marker=dict(size=8, color='blue'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=1, col=1
)

# HGNN Dimension (NEW)
fig_params.add_trace(
    go.Scatter(x=hparam_df['hgnn_dim'], y=hparam_df['test_f1'],
              mode='markers', name='HGNN Dim', marker=dict(size=8, color='purple'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=1, col=2
)

# Num HGNN Layers (NEW)
fig_params.add_trace(
    go.Scatter(x=hparam_df['num_hgnn_layers'], y=hparam_df['test_f1'],
              mode='markers', name='Num Layers', marker=dict(size=8, color='brown'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=1, col=3
)

# Attention Heads (NEW)
fig_params.add_trace(
    go.Scatter(x=hparam_df['attn_heads'], y=hparam_df['test_f1'],
              mode='markers', name='Attn Heads', marker=dict(size=8, color='teal'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=2, col=1
)

# Learning Rate (log scale)
fig_params.add_trace(
    go.Scatter(x=hparam_df['learning_rate'], y=hparam_df['test_f1'],
              mode='markers', name='Learning Rate', marker=dict(size=8, color='green'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=2, col=2
)

# Dropout
fig_params.add_trace(
    go.Scatter(x=hparam_df['dropout'], y=hparam_df['test_f1'],
              mode='markers', name='Dropout', marker=dict(size=8, color='red'),
              showlegend=False, text=hparam_df['trial'], hovertemplate='Trial %{text}<br>Test F1: %{y:.4f}'),
    row=2, col=3
)

# Update axes
fig_params.update_xaxes(title_text="Hidden Dimension", row=1, col=1)
fig_params.update_xaxes(title_text="HGNN Dimension", row=1, col=2)
fig_params.update_xaxes(title_text="Num HGNN Layers", row=1, col=3)
fig_params.update_xaxes(title_text="Attention Heads", row=2, col=1)
fig_params.update_xaxes(title_text="Learning Rate (log)", type='log', row=2, col=2)
fig_params.update_xaxes(title_text="Dropout", row=2, col=3)

for i in range(1, 7):
    row = (i - 1) // 3 + 1
    col = (i - 1) % 3 + 1
    fig_params.update_yaxes(title_text="Test F1", row=row, col=col)

fig_params.update_layout(height=900, width=1600, title_text="Hyperparameter Impact on Test F1 (7 parameters)")
fig_params.show()

# 3. Distribution of metrics
fig_dist = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Validation F1 Distribution', 'Test F1 Distribution', 'Test ROC-AUC Distribution'),
    specs=[[{'type': 'histogram'}, {'type': 'histogram'}, {'type': 'histogram'}]]
)

fig_dist.add_trace(go.Histogram(x=hparam_df['val_f1'], name='Val F1', nbinsx=15), row=1, col=1)
fig_dist.add_trace(go.Histogram(x=hparam_df['test_f1'], name='Test F1', nbinsx=15), row=1, col=2)
fig_dist.add_trace(go.Histogram(x=hparam_df['test_roc_auc'], name='Test ROC-AUC', nbinsx=15), row=1, col=3)

fig_dist.update_xaxes(title_text="Score", row=1, col=1)
fig_dist.update_xaxes(title_text="Score", row=1, col=2)
fig_dist.update_xaxes(title_text="Score", row=1, col=3)
fig_dist.update_yaxes(title_text="Frequency", row=1, col=1)

fig_dist.update_layout(height=400, width=1400, title_text="Metric Score Distributions",
                      showlegend=False)
fig_dist.show()

print("\n✅ Random search visualizations generated")

In [ ]:
# Generate recommendations and summary report
print("\n" + "="*80)
print("RANDOM SEARCH RECOMMENDATIONS & INSIGHTS")
print("="*80)

# Calculate correlations - EXPANDED for new hyperparameters
print("\n📊 PARAMETER CORRELATIONS WITH TEST F1:")
correlations = {
    'hidden_dim': hparam_df['hidden_dim'].corr(hparam_df['test_f1']),
    'hgnn_dim': hparam_df['hgnn_dim'].corr(hparam_df['test_f1']),
    'num_hgnn_layers': hparam_df['num_hgnn_layers'].corr(hparam_df['test_f1']),
    'attn_heads': hparam_df['attn_heads'].corr(hparam_df['test_f1']),
    'learning_rate': hparam_df['learning_rate'].corr(hparam_df['test_f1']),
    'weight_decay': hparam_df['weight_decay'].corr(hparam_df['test_f1']),
    'dropout': hparam_df['dropout'].corr(hparam_df['test_f1'])
}

for param, corr in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    strength = '(strong)' if abs(corr) > 0.5 else '(moderate)' if abs(corr) > 0.3 else '(weak)'
    print(f"   {param:20s}: {corr:7.4f} {strength}")

# Top parameter values analysis - EXPANDED for new hyperparameters
print("\n🔍 BEST PERFORMING PARAMETER VALUES:")

for param in ['hidden_dim', 'hgnn_dim', 'num_hgnn_layers', 'attn_heads', 'learning_rate', 'weight_decay', 'dropout']:
    top_3 = hparam_df.groupby(param)['test_f1'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False).head(3)
    print(f"\n   {param.upper()}:")
    for idx, (val, row) in enumerate(top_3.iterrows(), 1):
        print(f"      {idx}. {val:10} → Avg F1: {row['mean']:.4f} ± {row['std']:.4f} ({int(row['count'])} trials)")

# Recommendation
print(f"\n{'='*80}")
print(f"💡 RECOMMENDATIONS:")
print(f"{'='*80}")

best_config = hparam_df_sorted.iloc[0]
second_best = hparam_df_sorted.iloc[1] if len(hparam_df_sorted) > 1 else None

print(f"\n1. PRIMARY RECOMMENDATION (Best Trial):")
print(f"   Trial {best_config['trial']}: {best_config['experiment']}")
print(f"   Config: h={best_config['hidden_dim']}, lr={best_config['learning_rate']:.2e}, wd={best_config['weight_decay']:.2e}, dr={best_config['dropout']}")
print(f"   Performance: Val F1={best_config['val_f1']:.4f}, Test F1={best_config['test_f1']:.4f}, ROC-AUC={best_config['test_roc_auc']:.4f}")

if second_best is not None and abs(best_config['test_f1'] - second_best['test_f1']) < 0.01:
    print(f"\n2. ALTERNATIVE (Comparable Performance):")
    print(f"   Trial {second_best['trial']}: {second_best['experiment']}")
    print(f"   Config: h={second_best['hidden_dim']}, lr={second_best['learning_rate']:.2e}, wd={second_best['weight_decay']:.2e}, dr={second_best['dropout']}")
    print(f"   Performance: Val F1={second_best['val_f1']:.4f}, Test F1={second_best['test_f1']:.4f}, ROC-AUC={second_best['test_roc_auc']:.4f}")
    print(f"\n   Note: Similar performance to best config. May prefer for other reasons (speed, simplicity, etc.)")

print(f"\n3. INSIGHTS:")
print(f"   - Random search covered {len(hparam_df)} hyperparameter configurations")
print(f"   - Best improvement over baseline HGNN: +{(best_config['test_f1'] - hgnn_best_metrics['f1'])*100:.2f}% F1 Score")
print(f"   - Performance variance (std): {hparam_df['test_f1'].std():.4f}")
print(f"   - Best trial found at: Trial {best_config['trial']}/{len(param_combinations)}")

# Efficiency comparison
grid_size = 3 * 3 * 3 * 3  # Original grid search would have this many
print(f"\n4. SEARCH EFFICIENCY:")
print(f"   - Random search: {len(hparam_df)} trials")
print(f"   - Equivalent grid search: {grid_size} trials")
print(f"   - Efficiency gain: {(1 - len(hparam_df)/grid_size)*100:.1f}% fewer experiments")
print(f"   - Time saved: ~{(grid_size - len(hparam_df)) * 5} minutes (at 5 min/trial)")

print(f"\n{'='*80}\n")

In [34]:
# Visualize hyperparameter impact
print("\n📊 Generating hyperparameter analysis visualizations...")

# 1. Impact of each hyperparameter on Test F1 (fixed for continuous params)
fig_hparam_impact = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Hidden Dimension (box)',
        'Dropout (box)',
        'Learning Rate (log scatter)',
        'Weight Decay (log scatter)'
    ),
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# Hidden Dimension (categorical)
for hidden_dim in sorted(hparam_df['hidden_dim'].unique()):
    data = hparam_df[hparam_df['hidden_dim'] == hidden_dim]['test_f1']
    fig_hparam_impact.add_trace(
        go.Box(y=data, name=str(hidden_dim), showlegend=False),
        row=1, col=1
    )

# Dropout (categorical)
for dropout in sorted(hparam_df['dropout'].unique()):
    data = hparam_df[hparam_df['dropout'] == dropout]['test_f1']
    fig_hparam_impact.add_trace(
        go.Box(y=data, name=f"{dropout:.2f}", showlegend=False),
        row=1, col=2
    )

# Learning Rate (continuous)
lr = hparam_df['learning_rate'].values
f1_lr = hparam_df['test_f1'].values
fig_hparam_impact.add_trace(
    go.Scatter(x=lr, y=f1_lr, mode='markers',
               marker=dict(size=6, opacity=0.6, color='#1f77b4'),
               showlegend=False),
    row=2, col=1
)
# Trendline on log10(lr)
if len(lr) > 1:
    lr_log = np.log10(lr)
    m, b = np.polyfit(lr_log, f1_lr, 1)
    lr_line = np.logspace(lr_log.min(), lr_log.max(), 100)
    f1_line = m * np.log10(lr_line) + b
    fig_hparam_impact.add_trace(
        go.Scatter(x=lr_line, y=f1_line, mode='lines',
                   line=dict(color='black', width=2),
                   showlegend=False),
        row=2, col=1
    )

# Weight Decay (continuous)
wd = hparam_df['weight_decay'].values
f1_wd = hparam_df['test_f1'].values
fig_hparam_impact.add_trace(
    go.Scatter(x=wd, y=f1_wd, mode='markers',
               marker=dict(size=6, opacity=0.6, color='#ff7f0e'),
               showlegend=False),
    row=2, col=2
)
# Trendline on log10(wd)
if len(wd) > 1:
    wd_log = np.log10(wd)
    m, b = np.polyfit(wd_log, f1_wd, 1)
    wd_line = np.logspace(wd_log.min(), wd_log.max(), 100)
    f1_line = m * np.log10(wd_line) + b
    fig_hparam_impact.add_trace(
        go.Scatter(x=wd_line, y=f1_line, mode='lines',
                   line=dict(color='black', width=2),
                   showlegend=False),
        row=2, col=2
    )

fig_hparam_impact.update_yaxes(title_text="Test F1", row=1, col=1)
fig_hparam_impact.update_yaxes(title_text="Test F1", row=1, col=2)
fig_hparam_impact.update_yaxes(title_text="Test F1", row=2, col=1)
fig_hparam_impact.update_yaxes(title_text="Test F1", row=2, col=2)
fig_hparam_impact.update_xaxes(title_text="Hidden Dim", row=1, col=1)
fig_hparam_impact.update_xaxes(title_text="Dropout", row=1, col=2)
fig_hparam_impact.update_xaxes(title_text="Learning Rate (log)", type='log', row=2, col=1)
fig_hparam_impact.update_xaxes(title_text="Weight Decay (log)", type='log', row=2, col=2)

fig_hparam_impact.update_layout(
    height=900,
    width=1500,
    title_text="Hyperparameter Impact on Test F1 Score",
    showlegend=False
)

fig_hparam_impact.write_html(str(hparam_search_dir / "hyperparameter_impact.html"))
fig_hparam_impact.show()

print("✅ Saved: hyperparameter_impact.html")

# 2. Top 10 configurations comparison
fig_top10 = go.Figure()

top_10 = hparam_df_sorted.head(10)
colors_gradient = ['#2ca02c', '#3cb043', '#4cc05a', '#5cd071', '#6ce088',
                   '#7cf09f', '#8cffb6', '#9cffcd', '#acffe4', '#bcfffb']

for idx, (_, row) in enumerate(top_10.iterrows()):
    exp_short = row['experiment'].replace('hgnn_', '')
    fig_top10.add_trace(go.Bar(
        x=['F1', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall'],
        y=[row['test_f1'], row['test_roc_auc'], row['test_accuracy'],
           row['test_precision'], row['test_recall']],
        name=f"#{idx+1}: {exp_short[:30]}...",
        marker=dict(color=colors_gradient[idx])
    ))

fig_top10.update_layout(
    title="Top 10 HGNN Configurations - Test Metrics",
    barmode='group',
    height=600,
    width=1400,
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
    legend=dict(orientation="v", x=1.01, y=1)
)

fig_top10.write_html(str(hparam_search_dir / "top10_comparison.html"))
fig_top10.show()

print("✅ Saved: top10_comparison.html")

# 3. Heatmap: Learning Rate vs Hidden Dimension (Test F1)
pivot_lr_hidden = hparam_df.pivot_table(
    values='test_f1',
    index='learning_rate',
    columns='hidden_dim',
    aggfunc='mean'
)

fig_heatmap = go.Figure(data=go.Heatmap(
    z=pivot_lr_hidden.values,
    x=[str(c) for c in pivot_lr_hidden.columns],
    y=[f"{lr:.0e}" for lr in pivot_lr_hidden.index],
    colorscale='Viridis',
    text=np.round(pivot_lr_hidden.values, 4),
    texttemplate='%{text}',
    textfont={"size": 12},
    colorbar=dict(title="Test F1")
))

fig_heatmap.update_layout(
    title="Learning Rate vs Hidden Dimension (Average Test F1)",
    xaxis_title="Hidden Dimension",
    yaxis_title="Learning Rate",
    height=500,
    width=800
)

fig_heatmap.write_html(str(hparam_search_dir / "lr_hidden_heatmap.html"))
fig_heatmap.show()

print("✅ Saved: lr_hidden_heatmap.html")

print("\n✅ All hyperparameter analysis visualizations completed!")



📊 Generating hyperparameter analysis visualizations...


✅ Saved: hyperparameter_impact.html


NameError: name 'hparam_df_sorted' is not defined

In [ ]:
# Load and display top 10 hyperparameter configurations
import pandas as pd
from pathlib import Path

# Load the hyperparameter search results
hparam_search_dir = CHECKPOINTS_PATH / "model_comparison" / "hgnn_hyperparam_search"
hparam_csv = hparam_search_dir / "hyperparameter_search_results.csv"
print(hparam_csv)
if hparam_csv.exists():
    hparam_df = pd.read_csv(hparam_csv)
    hparam_df_sorted = hparam_df.sort_values('val_f1', ascending=False).reset_index(drop=True)
    
    print(f"Loaded hyperparameter search results from: {hparam_csv.name}")
    print(f"Total configurations tested: {len(hparam_df)}\n")
    print("="*120)
    print("TOP 10 HYPERPARAMETER CONFIGURATIONS (sorted by validation F1)")
    print("="*120)
    
    # Display relevant columns
    display_cols = ['trial', 'hidden_dim', 'hgnn_dim', 'num_hgnn_layers', 'attn_heads', 
                    'dropout', 'learning_rate', 'weight_decay', 
                    'val_f1', 'test_f1', 'test_roc_auc', 'test_accuracy']
    
    top_10 = hparam_df_sorted.head(10)[display_cols].copy()
    
    # Format for better readability
    top_10['learning_rate'] = top_10['learning_rate'].apply(lambda x: f"{x:.2e}")
    top_10['weight_decay'] = top_10['weight_decay'].apply(lambda x: f"{x:.2e}")
    
    print(top_10.to_string(index=True))
    
    print("\n" + "="*120)
    print("BEST CONFIGURATION DETAILS:")
    print("="*120)
    best = hparam_df_sorted.iloc[0]
    print(f"Trial: {best['trial']}")
    print(f"Experiment: {best['experiment']}")
    print(f"\nArchitecture:")
    print(f"  Hidden dim: {best['hidden_dim']}")
    print(f"  HGNN dim: {best['hgnn_dim']}")
    print(f"  HGNN layers: {best['num_hgnn_layers']}")
    print(f"  Attention heads: {best['attn_heads']}")
    print(f"  Dropout: {best['dropout']}")
    print(f"\nTraining:")
    print(f"  Learning rate: {best['learning_rate']:.2e}")
    print(f"  Weight decay: {best['weight_decay']:.2e}")
    print(f"\nPerformance:")
    print(f"  Validation F1: {best['val_f1']:.4f}")
    print(f"  Test F1: {best['test_f1']:.4f}")
    print(f"  Test ROC-AUC: {best['test_roc_auc']:.4f}")
    print(f"  Test Accuracy: {best['test_accuracy']:.4f}")
    print(f"  Test Precision: {best['test_precision']:.4f}")
    print(f"  Test Recall: {best['test_recall']:.4f}")
    
else:
    print(f"❌ CSV file not found: {hparam_csv}")
    print("Please run the hyperparameter search cells first.")

In [ ]:
# Deep validation of top 10 configurations - train multiple times for consistency
print("\n" + "="*80)
print("DEEP VALIDATION: TOP 10 CONFIGURATIONS")
print("Training each configuration 5 times to assess consistency")
print("="*80)

# Get top 10 configurations
top_10_configs = hparam_df_sorted.head(10).copy()

# Storage for multiple runs per configuration
top_10_multiple_runs = {}
top_10_consistency_results = []

num_runs_per_config = 5
extended_epochs = 50  # Train longer than initial 25 to ensure convergence

for idx, (_, config_row) in enumerate(top_10_configs.iterrows(), 1):
    config_trial = config_row['trial']
    config_name = config_row['experiment']
    
    print(f"\n{'='*80}")
    print(f"CONFIG {idx}/10 (Trial {config_trial}): {config_name}")
    print(f"{'='*80}")
    print(f"Architecture: h={config_row['hidden_dim']}, hd={config_row['hgnn_dim']}, "
          f"nl={config_row['num_hgnn_layers']}, ah={config_row['attn_heads']}")
    print(f"Training: lr={config_row['learning_rate']:.2e}, wd={config_row['weight_decay']:.2e}, "
          f"drop={config_row['dropout']}")
    print(f"Initial results: Val F1={config_row['val_f1']:.4f}, Test F1={config_row['test_f1']:.4f}, "
          f"ROC-AUC={config_row['test_roc_auc']:.4f}\n")
    
    run_results = []
    
    for run_idx in range(1, num_runs_per_config + 1):
        print(f"  Run {run_idx}/{num_runs_per_config}...", end=' ')
        run_start = time.time()
        
        # Create fresh model instance
        model_validation = HGNNOutfitScorer(
            in_dim=X_train_combined_t.shape[1],
            hidden_dim=int(config_row['hidden_dim']),
            hgnn_dim=int(config_row['hgnn_dim']),
            num_hgnn_layers=int(config_row['num_hgnn_layers']),
            attn_heads=int(config_row['attn_heads']),
            dropout=float(config_row['dropout'])
        )
        
        # Train with extended epochs
        model_validation, best_val_loss_v, best_metrics_v, history_v = train_model(
            model=model_validation,
            train_loader=train_loader,
            val_loader=val_loader,
            adj_train=None,
            H_train=H_train,
            adj_val=None,
            H_val=H_val,
            epochs=extended_epochs,
            lr=float(config_row['learning_rate']),
            model_name=f"validation_run_{run_idx}",
            early_stopping_patience=7,
            early_stopping_min_delta=1e-4
        )
        
        # Evaluate on test set
        test_metrics_v, _, _, _ = evaluate_model(
            model_validation, test_loader, X_test_combined_t, N_test, 
            f"validation_run_{run_idx}", is_hgnn=True
        )
        
        run_duration = time.time() - run_start
        
        run_results.append({
            'run': run_idx,
            'val_f1': float(best_metrics_v.get('f1', max(history_v['f1']) if history_v['f1'] else 0.0)),
            'val_loss': float(best_val_loss_v),
            'test_f1': float(test_metrics_v['f1']),
            'test_roc_auc': float(test_metrics_v['roc_auc']),
            'test_accuracy': float(test_metrics_v['accuracy']),
            'test_precision': float(test_metrics_v['precision']),
            'test_recall': float(test_metrics_v['recall']),
            'epochs_trained': len(history_v['train_loss']),
            'duration_sec': run_duration
        })
        
        print(f"✓ Test F1={test_metrics_v['f1']:.4f}, ROC-AUC={test_metrics_v['roc_auc']:.4f}, "
              f"Epochs={len(history_v['train_loss'])}, Time={run_duration:.1f}s")
        
        del model_validation
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Calculate statistics for this configuration
    run_df = pd.DataFrame(run_results)
    
    consistency_summary = {
        'rank': idx,
        'trial': config_trial,
        'experiment': config_name,
        'initial_test_f1': float(config_row['test_f1']),
        # Validation F1 statistics
        'val_f1_mean': float(run_df['val_f1'].mean()),
        'val_f1_std': float(run_df['val_f1'].std()),
        'val_f1_min': float(run_df['val_f1'].min()),
        'val_f1_max': float(run_df['val_f1'].max()),
        # Test F1 statistics
        'test_f1_mean': float(run_df['test_f1'].mean()),
        'test_f1_std': float(run_df['test_f1'].std()),
        'test_f1_min': float(run_df['test_f1'].min()),
        'test_f1_max': float(run_df['test_f1'].max()),
        # ROC-AUC statistics
        'test_roc_auc_mean': float(run_df['test_roc_auc'].mean()),
        'test_roc_auc_std': float(run_df['test_roc_auc'].std()),
        'test_roc_auc_min': float(run_df['test_roc_auc'].min()),
        'test_roc_auc_max': float(run_df['test_roc_auc'].max()),
        # Other metrics (mean across runs)
        'test_accuracy': float(run_df['test_accuracy'].mean()),
        'test_precision': float(run_df['test_precision'].mean()),
        'test_recall': float(run_df['test_recall'].mean()),
        # Consistency metric (lower std = more consistent)
        'consistency_score': float(1.0 / (1.0 + run_df['test_f1'].std())),  # Higher is more consistent
        'avg_epochs': float(run_df['epochs_trained'].mean()),
        'avg_duration_sec': float(run_df['duration_sec'].mean())
    }
    
    top_10_multiple_runs[config_name] = run_df
    top_10_consistency_results.append(consistency_summary)
    
    print(f"\n  Summary:")
    print(f"    Test F1:     {consistency_summary['test_f1_mean']:.4f} ± {consistency_summary['test_f1_std']:.4f} "
          f"(range: {consistency_summary['test_f1_min']:.4f}-{consistency_summary['test_f1_max']:.4f})")
    print(f"    ROC-AUC:     {consistency_summary['test_roc_auc_mean']:.4f} ± {consistency_summary['test_roc_auc_std']:.4f} "
          f"(range: {consistency_summary['test_roc_auc_min']:.4f}-{consistency_summary['test_roc_auc_max']:.4f})")
    print(f"    Consistency: {consistency_summary['consistency_score']:.4f} (higher = more stable)")
    print(f"    Avg Epochs:  {consistency_summary['avg_epochs']:.1f}")

# Create summary dataframe
consistency_df = pd.DataFrame(top_10_consistency_results)
consistency_df_sorted = consistency_df.sort_values('test_f1_mean', ascending=False).reset_index(drop=True)

print(f"\n{'='*80}")
print("TOP 10 CONFIGURATION CONSISTENCY RANKING")
print(f"{'='*80}\n")

display_cols = ['rank', 'trial', 'test_f1_mean', 'test_f1_std', 'test_roc_auc_mean', 
                'test_roc_auc_std', 'consistency_score', 'avg_epochs']
print(consistency_df_sorted[display_cols].to_string(index=False))

# Identify the most stable configuration
best_stability_idx = consistency_df_sorted['consistency_score'].idxmax()
most_stable = consistency_df_sorted.iloc[best_stability_idx]

print(f"\n{'='*80}")
print(f"✨ MOST STABLE & CONSISTENT CONFIGURATION")
print(f"{'='*80}")
print(f"Rank: {most_stable['rank']}")
print(f"Trial: {most_stable['trial']}")
print(f"Experiment: {most_stable['experiment']}")
print(f"\nPerformance (across {num_runs_per_config} runs):")
print(f"  Test F1:     {most_stable['test_f1_mean']:.4f} ± {most_stable['test_f1_std']:.4f}")
print(f"  ROC-AUC:     {most_stable['test_roc_auc_mean']:.4f} ± {most_stable['test_roc_auc_std']:.4f}")
print(f"  Accuracy:    {most_stable['test_accuracy']:.4f}")
print(f"  Precision:   {most_stable['test_precision']:.4f}")
print(f"  Recall:      {most_stable['test_recall']:.4f}")
print(f"\nConsistency Score: {most_stable['consistency_score']:.4f}")
print(f"Average Training: {most_stable['avg_epochs']:.1f} epochs, {most_stable['avg_duration_sec']:.1f} sec/run")

# Save consistency results
consistency_csv = hparam_search_dir / "top_10_consistency_analysis.csv"
consistency_df_sorted.to_csv(consistency_csv, index=False)
print(f"\n✅ Consistency analysis saved: {consistency_csv.name}")

# Create visualizations
fig_consistency = go.Figure()

# Plot 1: Test F1 mean ± std with ranges
fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_mean'],
    error_y=dict(
        type='data',
        array=consistency_df_sorted['test_f1_std'],
        visible=True
    ),
    mode='lines+markers',
    name='Test F1 (mean ± std)',
    marker=dict(size=10, color='darkblue'),
    line=dict(width=2)
))

# Add min/max range
fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_max'],
    fill=None,
    mode='lines',
    line_color='rgba(0,0,0,0)',
    showlegend=False
))

fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_min'],
    fillcolor='rgba(0,100,200,0.2)',
    fill='tonexty',
    mode='lines',
    line_color='rgba(0,0,0,0)',
    name='Min-Max Range'
))

fig_consistency.update_layout(
    title=f"Top 10 Configurations: Consistency Analysis ({num_runs_per_config} runs each)",
    xaxis_title="Configuration Rank",
    yaxis_title="Test F1 Score",
    height=500,
    width=1000,
    hovermode='x unified',
    template='plotly_white'
)

consistency_html = hparam_search_dir / f"top_10_consistency_analysis.html"
fig_consistency.write_html(str(consistency_html))
print(f"✅ Visualization saved: {consistency_html.name}")

In [ ]:

# Create summary dataframe
consistency_df = pd.DataFrame(top_10_consistency_results)
consistency_df_sorted = consistency_df.sort_values('test_f1_mean', ascending=False).reset_index(drop=True)

print(f"\n{'='*80}")
print("TOP 10 CONFIGURATION CONSISTENCY RANKING")
print(f"{'='*80}\n")

display_cols = ['rank', 'trial', 'test_f1_mean', 'test_f1_std', 'test_roc_auc_mean', 
                'test_roc_auc_std', 'consistency_score', 'avg_epochs']
print(consistency_df_sorted[display_cols].to_string(index=False))

# Identify the most stable configuration
best_stability_idx = consistency_df_sorted['consistency_score'].idxmax()
most_stable = consistency_df_sorted.iloc[best_stability_idx]

print(f"\n{'='*80}")
print(f"✨ MOST STABLE & CONSISTENT CONFIGURATION")
print(f"{'='*80}")
print(f"Rank: {most_stable['rank']}")
print(f"Trial: {most_stable['trial']}")
print(f"Experiment: {most_stable['experiment']}")
print(f"\nPerformance (across {num_runs_per_config} runs):")
print(f"  Test F1:     {most_stable['test_f1_mean']:.4f} ± {most_stable['test_f1_std']:.4f}")
print(f"  ROC-AUC:     {most_stable['test_roc_auc_mean']:.4f} ± {most_stable['test_roc_auc_std']:.4f}")
print(f"\nConsistency Score: {most_stable['consistency_score']:.4f}")
print(f"Average Training: {most_stable['avg_epochs']:.1f} epochs, {most_stable['avg_duration_sec']:.1f} sec/run")

# Save consistency results
consistency_csv = hparam_search_dir / "top_10_consistency_analysis.csv"
consistency_df_sorted.to_csv(consistency_csv, index=False)
print(f"\n✅ Consistency analysis saved: {consistency_csv.name}")

# Create visualizations
fig_consistency = go.Figure()

# Plot 1: Test F1 mean ± std with ranges
fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_mean'],
    error_y=dict(
        type='data',
        array=consistency_df_sorted['test_f1_std'],
        visible=True
    ),
    mode='lines+markers',
    name='Test F1 (mean ± std)',
    marker=dict(size=10, color='darkblue'),
    line=dict(width=2)
))

# Add min/max range
fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_max'],
    fill=None,
    mode='lines',
    line_color='rgba(0,0,0,0)',
    showlegend=False
))

fig_consistency.add_trace(go.Scatter(
    x=consistency_df_sorted['rank'],
    y=consistency_df_sorted['test_f1_min'],
    fillcolor='rgba(0,100,200,0.2)',
    fill='tonexty',
    mode='lines',
    line_color='rgba(0,0,0,0)',
    name='Min-Max Range'
))

fig_consistency.update_layout(
    title=f"Top 10 Configurations: Consistency Analysis ({num_runs_per_config} runs each)",
    xaxis_title="Configuration Rank",
    yaxis_title="Test F1 Score",
    height=500,
    width=1000,
    hovermode='x unified',
    template='plotly_white'
)

consistency_html = hparam_search_dir / f"top_10_consistency_analysis.html"
fig_consistency.write_html(str(consistency_html))
print(f"✅ Visualization saved: {consistency_html.name}")

In [ ]:
fig_top10.show()

# 13. Final Model Training - Best Configuration Until Convergence

Train the best performing configuration with extended training until full convergence

In [15]:
# Select the best configuration based on consistency analysis
print("\n" + "="*80)
print("FINAL MODEL TRAINING - BEST CONFIGURATION")
print("="*80)

checkpoint_base_dir = CHECKPOINTS_PATH / "model_comparison"

# Load results from CSV files
hparam_search_dir = checkpoint_base_dir / "hgnn_hyperparam_search"

# Try to load consistency analysis CSV first (from deep validation)
consistency_csv = hparam_search_dir / "top_10_consistency_analysis.csv"
hparam_csv = hparam_search_dir / "hyperparameter_search_results.csv"

if consistency_csv.exists():
    print(f"\n📂 Loading consistency analysis from: {consistency_csv.name}")
    consistency_df = pd.read_csv(consistency_csv)
    
    # Determine best configuration: balance between performance and consistency
    # Score = 0.6 * test_f1_mean + 0.4 * consistency_score
    consistency_df['final_score'] = (
        0.6 * consistency_df['test_f1_mean'] + 
        0.4 * consistency_df['consistency_score']
    )
    
    consistency_df_sorted = consistency_df.sort_values('final_score', ascending=False).reset_index(drop=True)
    best_overall_idx = consistency_df_sorted['final_score'].idxmax()
    best_final = consistency_df_sorted.iloc[best_overall_idx]
    
    print(f"\n✨ SELECTED CONFIGURATION FOR FINAL TRAINING:")
    print(f"{'='*80}")
    print(f"Rank: {best_final['rank']}")
    print(f"Trial: {best_final['trial']}")
    print(f"Experiment: {best_final['experiment']}")
    print(f"\nPerformance (from consistency validation runs):")
    print(f"  Test F1:     {best_final['test_f1_mean']:.4f} ± {best_final['test_f1_std']:.4f}")
    print(f"  ROC-AUC:     {best_final['test_roc_auc_mean']:.4f} ± {best_final['test_roc_auc_std']:.4f}")
    print(f"  Consistency: {best_final['consistency_score']:.4f}")
    print(f"  Final Score: {best_final['final_score']:.4f}")
    
    # Get the original hyperparameters from the initial search
    best_trial = int(best_final['trial'])
    
elif hparam_csv.exists():
    print(f"\n📂 Loading hyperparameter search from: {hparam_csv.name}")
    print("   (Consistency analysis not found, using initial search results)")
    hparam_df = pd.read_csv(hparam_csv)
    hparam_df_sorted = hparam_df.sort_values('val_f1', ascending=False).reset_index(drop=True)
    
    best_final = hparam_df_sorted.iloc[0]
    best_trial = int(best_final['trial'])
    
    print(f"\n✨ SELECTED CONFIGURATION FOR FINAL TRAINING:")
    print(f"{'='*80}")
    print(f"Trial: {best_trial}")
    print(f"Experiment: {best_final['experiment']}")
    print(f"\nPerformance (from initial search):")
    print(f"  Val F1:  {best_final['val_f1']:.4f}")
    print(f"  Test F1: {best_final['test_f1']:.4f}")
    
else:
    raise FileNotFoundError(
        f"❌ Could not find hyperparameter search results!\n"
        f"   Expected files:\n"
        f"   - {consistency_csv}\n"
        f"   - {hparam_csv}\n"
        f"   Please run the hyperparameter search cells first."
    )

# Load hyperparameters from initial search CSV
print(f"\n📂 Loading hyperparameters from: {hparam_csv.name}")
hparam_df = pd.read_csv(hparam_csv)
best_config_row = hparam_df[hparam_df['trial'] == best_trial].iloc[0]

print(f"\nHyperparameters:")
print(f"  Architecture:")
print(f"    - Hidden dim:    {int(best_config_row['hidden_dim'])}")
print(f"    - HGNN dim:      {int(best_config_row['hgnn_dim'])}")
print(f"    - HGNN layers:   {int(best_config_row['num_hgnn_layers'])}")
print(f"    - Attn heads:    {int(best_config_row['attn_heads'])}")
print(f"    - Dropout:       {float(best_config_row['dropout'])}")
print(f"  Training:")
print(f"    - Learning rate: {float(best_config_row['learning_rate']):.4e}")
print(f"    - Weight decay:  {float(best_config_row['weight_decay']):.4e}")

# Store configuration for final training
final_config = {
    'trial': best_trial,
    'experiment': best_config_row['experiment'],
    'hidden_dim': int(best_config_row['hidden_dim']),
    'hgnn_dim': int(best_config_row['hgnn_dim']),
    'num_hgnn_layers': int(best_config_row['num_hgnn_layers']),
    'attn_heads': int(best_config_row['attn_heads']),
    'dropout': float(best_config_row['dropout']),
    'learning_rate': float(best_config_row['learning_rate']),
    'weight_decay': float(best_config_row['weight_decay'])
}

print(f"\n{'='*80}")


FINAL MODEL TRAINING - BEST CONFIGURATION

📂 Loading consistency analysis from: top_10_consistency_analysis.csv

✨ SELECTED CONFIGURATION FOR FINAL TRAINING:
Rank: 1
Trial: 22
Experiment: hgnn_trial22_h256_hd32_nl1_ah4_lr1.08e-03_wd1.79e-04_dr30

Performance (from consistency validation runs):
  Test F1:     0.8668 ± 0.0061
  ROC-AUC:     0.9427 ± 0.0038
  Consistency: 0.9939
  Final Score: 0.9177

📂 Loading hyperparameters from: hyperparameter_search_results.csv

Hyperparameters:
  Architecture:
    - Hidden dim:    256
    - HGNN dim:      32
    - HGNN layers:   1
    - Attn heads:    4
    - Dropout:       0.3
  Training:
    - Learning rate: 1.0824e-03
    - Weight decay:  1.7885e-04



In [17]:
import time

In [24]:
# Create and train final model until convergence
print("\n" + "="*80)
print("TRAINING FINAL MODEL UNTIL CONVERGENCE")
print("="*80)

# Extended training configuration
final_epochs = 100  # Much longer to ensure full convergence
final_patience = 15  # Higher patience to avoid premature stopping
final_min_delta = 1e-5  # Smaller delta for finer convergence detection

print(f"\nTraining Configuration:")
print(f"  Max epochs:       {final_epochs}")
print(f"  Early stopping:   patience={final_patience}, min_delta={final_min_delta}")
print(f"  Batch size:       {batch_size}")
print(f"  Learning rate:    {final_config['learning_rate']:.4e}")
print(f"  Weight decay:     {final_config['weight_decay']:.4e}")

# Create final model instance
final_model = HGNNOutfitScorer(
    in_dim=X_train_combined_t.shape[1],
    hidden_dim=final_config['hidden_dim'],
    hgnn_dim=final_config['hgnn_dim'],
    num_hgnn_layers=final_config['num_hgnn_layers'],
    attn_heads=final_config['attn_heads'],
    dropout=final_config['dropout']
)

print(f"\n🚀 Starting final training...")
start_final_training = time.time()

# Train the final model
final_model, final_best_val_loss, final_best_metrics, final_history = train_model(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    adj_train=None,
    H_train=H_train,
    adj_val=None,
    H_val=H_val,
    epochs=final_epochs,
    lr=final_config['learning_rate'],
    model_name="FINAL_HGNN",
    early_stopping_patience=final_patience,
    early_stopping_min_delta=final_min_delta
)

final_training_duration = time.time() - start_final_training

print(f"\n✅ Final training completed!")
print(f"   Duration: {final_training_duration/60:.2f} minutes")
print(f"   Epochs trained: {len(final_history['train_loss'])}")
print(f"   Best validation loss: {final_best_val_loss:.4f}")
print(f"   Best validation F1: {final_best_metrics.get('f1', max(final_history['f1'])):.4f}")


TRAINING FINAL MODEL UNTIL CONVERGENCE

Training Configuration:
  Max epochs:       100
  Early stopping:   patience=15, min_delta=1e-05
  Batch size:       1024
  Learning rate:    1.0824e-03
  Weight decay:     1.7885e-04

🚀 Starting final training...

Training FINAL_HGNN on cuda | Epochs: 100 | Batches/epoch: 34
Early Stopping: patience=15, min_delta=1e-05
Feature shapes: Train Xc=torch.Size([202110, 512]), Xa=torch.Size([202110, 256]), Combined=torch.Size([202110, 768])
Feature shapes: Val   Xc=torch.Size([202110, 512]), Xa=torch.Size([202110, 256]), Combined=torch.Size([202110, 768])
Using training incidence matrix: torch.Size([202110, 33990])
Using validation incidence matrix: torch.Size([202110, 6000])



Epoch 1/100 | TrainLoss=0.675529 | ValLoss=0.638596 | BestVal=0.638596 | MSE=0.223618 | ROC-AUC=0.6936 | F1=0.5961


Epoch 2/100 | TrainLoss=0.608671 | ValLoss=0.484929 | BestVal=0.484929 | MSE=0.156772 | ROC-AUC=0.8602 | F1=0.7690


Epoch 3/100 | TrainLoss=0.471100 | ValLoss=0.372406 | BestVal=0.372406 | MSE=0.114622 | ROC-AUC=0.9176 | F1=0.8519


Epoch 4/100 | TrainLoss=0.407031 | ValLoss=0.316232 | BestVal=0.316232 | MSE=0.094768 | ROC-AUC=0.9397 | F1=0.8682


...

Epoch 28/100 | TrainLoss=0.143753 | ValLoss=0.274265 | BestVal=0.240559 | MSE=0.070113 | ROC-AUC=0.9687 | F1=0.9115


Epoch 29/100 | TrainLoss=0.139672 | ValLoss=0.296973 | BestVal=0.240559 | MSE=0.070605 | ROC-AUC=0.9680 | F1=0.9150


Epoch 30/100 | TrainLoss=0.140870 | ValLoss=0.278680 | BestVal=0.240559 | MSE=0.068530 | ROC-AUC=0.9703 | F1=0.9156


Epoch 31/100 | TrainLoss=0.138527 | ValLoss=0.290551 | BestVal=0.240559 | MSE=0.071609 | ROC-AUC=0.9686 | F1=0.9128


Epoch 32/100 | TrainLoss=0.131064 | ValLoss=0.303276 | BestVal=0.240559 | MSE=0.072822 | ROC-AUC=0.9671 | F1=0.9097

Early stopping at epoch 32. Best val_loss: 0.240559

✅ Loaded best model (Val Loss: 0.240559)

✅ Final training completed!
   Duration: 36.94 minutes
   Epochs trained: 32
   Best validation loss: 0.2406
   Best validation F1: 0.9130


In [25]:
# Evaluate final model on all datasets
print("\n" + "="*80)
print("FINAL MODEL EVALUATION")
print("="*80)

# Evaluate on training set
print("\n📊 Evaluating on TRAINING set...")
final_train_metrics, final_train_probs, final_train_preds, final_train_labels = evaluate_model(
    final_model, train_loader, N_train,
    "FINAL_HGNN_train", H=H_train, is_hgnn=True
)

# Evaluate on validation set
print("\n📊 Evaluating on VALIDATION set...")
final_val_metrics, final_val_probs, final_val_preds, final_val_labels = evaluate_model(
    final_model, val_loader, N_val,
    "FINAL_HGNN_val", H=H_val, is_hgnn=True
)

# Evaluate on test set
print("\n📊 Evaluating on TEST set...")
final_test_metrics, final_test_probs, final_test_preds, final_test_labels = evaluate_model(
    final_model, test_loader, N_test,
    "FINAL_HGNN_test", H=H_test, is_hgnn=True
)



# Display comprehensive results
print("\n" + "="*80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*80)

results_summary = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'Loss': [final_history['train_loss'][-1], final_best_val_loss, 0.0],
    'Accuracy': [final_train_metrics['accuracy'], final_val_metrics['accuracy'], final_test_metrics['accuracy']],
    'Precision': [final_train_metrics['precision'], final_val_metrics['precision'], final_test_metrics['precision']],
    'Recall': [final_train_metrics['recall'], final_val_metrics['recall'], final_test_metrics['recall']],
    'F1': [final_train_metrics['f1'], final_val_metrics['f1'], final_test_metrics['f1']],
    'ROC-AUC': [final_train_metrics['roc_auc'], final_val_metrics['roc_auc'], final_test_metrics['roc_auc']],
    'MSE': [final_train_metrics['mse'], final_val_metrics['mse'], final_test_metrics['mse']]
})

print("\n" + results_summary.to_string(index=False))

# Calculate generalization gaps
train_val_gap = (final_train_metrics['f1'] - final_val_metrics['f1']) * 100
val_test_gap = (final_val_metrics['f1'] - final_test_metrics['f1']) * 100
train_test_gap = (final_train_metrics['f1'] - final_test_metrics['f1']) * 100

print(f"\n📉 Generalization Analysis:")
print(f"   Train → Val gap:  {train_val_gap:+.2f}% F1")
print(f"   Val → Test gap:   {val_test_gap:+.2f}% F1")
print(f"   Train → Test gap: {train_test_gap:+.2f}% F1")

if abs(val_test_gap) < 2.0:
    print(f"   ✅ Excellent generalization (gap < 2%)")
elif abs(val_test_gap) < 5.0:
    print(f"   ✓ Good generalization (gap < 5%)")
else:
    print(f"   ⚠️  Moderate generalization gap")

print("\n" + "="*80)



FINAL MODEL EVALUATION

📊 Evaluating on TRAINING set...

📊 Evaluating on VALIDATION set...

📊 Evaluating on TEST set...

FINAL MODEL PERFORMANCE SUMMARY

   Dataset     Loss  Accuracy  Precision   Recall       F1  ROC-AUC      MSE
  Training 0.131064  0.968991   0.970541 0.967343 0.968940 0.993885 0.024865
Validation 0.240559  0.913500   0.918664 0.907333 0.912963 0.969132 0.067597
      Test 0.000000  0.874480   0.901238 0.841136 0.870150 0.943271 0.097021

📉 Generalization Analysis:
   Train → Val gap:  +5.60% F1
   Val → Test gap:   +4.28% F1
   Train → Test gap: +9.88% F1
   ✓ Good generalization (gap < 5%)



In [27]:
final_train_metrics

{'model': 'FINAL_HGNN_train',
 'accuracy': 0.9689908796704914,
 'precision': 0.9705413542712085,
 'recall': 0.9673433362753752,
 'f1': 0.968939706489067,
 'roc_auc': 0.9938848462863235,
 'mse': 0.024865418672561646,
 'tp': np.int64(16440),
 'fp': np.int64(499),
 'tn': np.int64(16496),
 'fn': np.int64(555)}

In [28]:
# Save final model and create comprehensive report
print("\n" + "="*80)
print("SAVING FINAL MODEL")
print("="*80)

# Create final model directory
final_model_dir = checkpoint_base_dir / "final_model"
final_model_dir.mkdir(parents=True, exist_ok=True)

timestamp_final = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Save model weights
model_path = final_model_dir / f"best_hgnn_model_{timestamp_final}.pt"
torch.save(final_model.state_dict(), model_path)
print(f"✅ Model weights saved: {model_path.name}")

# 2. Save complete model (architecture + weights)
complete_model_path = final_model_dir / f"best_hgnn_complete_{timestamp_final}.pt"
torch.save({
    'model_state_dict': final_model.state_dict(),
    'model_config': final_config,
    'input_dim': X_train_combined_t.shape[1],
    'training_config': {
        'batch_size': batch_size,
        'max_epochs': final_epochs,
        'actual_epochs': len(final_history['train_loss']),
        'learning_rate': final_config['learning_rate'],
        'weight_decay': final_config['weight_decay'],
        'patience': final_patience,
        'min_delta': final_min_delta
    }
}, complete_model_path)
print(f"✅ Complete model saved: {complete_model_path.name}")

# 3. Save training history
history_final = {
    'train_loss': [float(x) for x in final_history['train_loss']],
    'val_loss': [float(x) for x in final_history['val_loss']],
    'mse': [float(x) for x in final_history['mse']],
    'roc_auc': [float(x) for x in final_history['roc_auc']],
    'accuracy': [float(x) for x in final_history['accuracy']],
    'f1': [float(x) for x in final_history['f1']],
    'epochs': len(final_history['train_loss']),
    'duration_minutes': final_training_duration / 60
}

history_path = final_model_dir / f"training_history_{timestamp_final}.json"
with open(history_path, 'w') as f:
    json.dump(history_final, f, indent=2)
print(f"✅ Training history saved: {history_path.name}")

# Helper: keep only numeric metrics

def _numeric_metrics(metrics_dict):
    return {k: float(v) for k, v in metrics_dict.items() if isinstance(v, (int, float, np.number))}

# 4. Save performance metrics
performance_report = {
    'model_name': 'HGNN_Final',
    'timestamp': timestamp_final,
    'configuration': final_config,
    'training': {
        'epochs_trained': len(final_history['train_loss']),
        'max_epochs': final_epochs,
        'duration_minutes': final_training_duration / 60,
        'early_stopped': len(final_history['train_loss']) < final_epochs,
        'best_val_loss': float(final_best_val_loss)
    },
    'performance': {
        'train': _numeric_metrics(final_train_metrics),
        'validation': _numeric_metrics(final_val_metrics),
        'test': _numeric_metrics(final_test_metrics)
    },
    'generalization': {
        'train_val_gap_pct': float(train_val_gap),
        'val_test_gap_pct': float(val_test_gap),
        'train_test_gap_pct': float(train_test_gap)
    },
    'comparison_to_initial_search': {
        'initial_test_f1': float(best_config_row['test_f1']),
        'final_test_f1': float(final_test_metrics['f1']),
        'improvement_pct': float((final_test_metrics['f1'] - best_config_row['test_f1']) * 100)
    }
}

metrics_path = final_model_dir / f"performance_report_{timestamp_final}.json"
with open(metrics_path, 'w') as f:
    json.dump(performance_report, f, indent=2)
print(f"✅ Performance report saved: {metrics_path.name}")

# 5. Save configuration as YAML-like readable format
config_path = final_model_dir / f"model_config_{timestamp_final}.txt"
with open(config_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("FINAL HGNN MODEL CONFIGURATION\n")
    f.write("="*60 + "\n\n")
    f.write(f"Timestamp: {timestamp_final}\n")
    f.write(f"Trial: {final_config['trial']}\n")
    f.write(f"Experiment: {final_config['experiment']}\n\n")
    
    f.write("ARCHITECTURE:\n")
    f.write(f"  Input Dimension:     {X_train_combined_t.shape[1]}\n")
    f.write(f"  Hidden Dimension:    {final_config['hidden_dim']}\n")
    f.write(f"  HGNN Dimension:      {final_config['hgnn_dim']}\n")
    f.write(f"  HGNN Layers:         {final_config['num_hgnn_layers']}\n")
    f.write(f"  Attention Heads:     {final_config['attn_heads']}\n")
    f.write(f"  Dropout:             {final_config['dropout']}\n\n")
    
    f.write("TRAINING:\n")
    f.write(f"  Learning Rate:       {final_config['learning_rate']:.6f}\n")
    f.write(f"  Weight Decay:        {final_config['weight_decay']:.6f}\n")
    f.write(f"  Batch Size:          {batch_size}\n")
    f.write(f"  Epochs (max):        {final_epochs}\n")
    f.write(f"  Epochs (actual):     {len(final_history['train_loss'])}\n")
    f.write(f"  Early Stop Patience: {final_patience}\n")
    f.write(f"  Duration:            {final_training_duration/60:.2f} min\n\n")
    
    f.write("PERFORMANCE:\n")
    f.write(f"  Test F1:             {final_test_metrics['f1']:.4f}\n")
    f.write(f"  Test ROC-AUC:        {final_test_metrics['roc_auc']:.4f}\n")
    f.write(f"  Test Accuracy:       {final_test_metrics['accuracy']:.4f}\n")
    f.write(f"  Test Precision:      {final_test_metrics['precision']:.4f}\n")
    f.write(f"  Test Recall:         {final_test_metrics['recall']:.4f}\n\n")
    
    f.write("GENERALIZATION:\n")
    f.write(f"  Train-Val Gap:       {train_val_gap:+.2f}%\n")
    f.write(f"  Val-Test Gap:        {val_test_gap:+.2f}%\n")
    f.write(f"  Train-Test Gap:      {train_test_gap:+.2f}%\n")

print(f"✅ Configuration file saved: {config_path.name}")



SAVING FINAL MODEL
✅ Model weights saved: best_hgnn_model_20260129_090704.pt
✅ Complete model saved: best_hgnn_complete_20260129_090704.pt
✅ Training history saved: training_history_20260129_090704.json
✅ Performance report saved: performance_report_20260129_090704.json
✅ Configuration file saved: model_config_20260129_090704.txt


In [29]:
# Visualize final model training curves
print("\n📊 Creating final model visualizations...")

fig_final = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Training & Validation Loss', 'F1 Score Progression', 
                    'ROC-AUC Progression', 'All Metrics Comparison'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

epochs_range = list(range(1, len(final_history['train_loss']) + 1))

# 1. Loss curves
fig_final.add_trace(
    go.Scatter(x=epochs_range, y=final_history['train_loss'], 
               name='Train Loss', mode='lines', line=dict(color='blue', width=2)),
    row=1, col=1
)
fig_final.add_trace(
    go.Scatter(x=epochs_range, y=final_history['val_loss'], 
               name='Val Loss', mode='lines', line=dict(color='red', width=2)),
    row=1, col=1
)

# 2. F1 Score
fig_final.add_trace(
    go.Scatter(x=epochs_range, y=final_history['f1'], 
               name='F1 Score', mode='lines+markers', 
               line=dict(color='green', width=2), marker=dict(size=4)),
    row=1, col=2
)

# 3. ROC-AUC
fig_final.add_trace(
    go.Scatter(x=epochs_range, y=final_history['roc_auc'], 
               name='ROC-AUC', mode='lines+markers', 
               line=dict(color='purple', width=2), marker=dict(size=4)),
    row=2, col=1
)

# 4. Final metrics comparison across datasets
metrics_comp = ['F1', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall']
train_vals = [final_train_metrics['f1'], final_train_metrics['roc_auc'], 
              final_train_metrics['accuracy'], final_train_metrics['precision'], 
              final_train_metrics['recall']]
val_vals = [final_val_metrics['f1'], final_val_metrics['roc_auc'], 
            final_val_metrics['accuracy'], final_val_metrics['precision'], 
            final_val_metrics['recall']]
test_vals = [final_test_metrics['f1'], final_test_metrics['roc_auc'], 
             final_test_metrics['accuracy'], final_test_metrics['precision'], 
             final_test_metrics['recall']]

fig_final.add_trace(
    go.Bar(x=metrics_comp, y=train_vals, name='Train', marker_color='lightblue'),
    row=2, col=2
)
fig_final.add_trace(
    go.Bar(x=metrics_comp, y=val_vals, name='Validation', marker_color='lightcoral'),
    row=2, col=2
)
fig_final.add_trace(
    go.Bar(x=metrics_comp, y=test_vals, name='Test', marker_color='lightgreen'),
    row=2, col=2
)

# Update layout
fig_final.update_xaxes(title_text="Epoch", row=1, col=1)
fig_final.update_xaxes(title_text="Epoch", row=1, col=2)
fig_final.update_xaxes(title_text="Epoch", row=2, col=1)
fig_final.update_xaxes(title_text="Metric", row=2, col=2)

fig_final.update_yaxes(title_text="Loss", row=1, col=1)
fig_final.update_yaxes(title_text="F1 Score", row=1, col=2, range=[0, 1])
fig_final.update_yaxes(title_text="ROC-AUC", row=2, col=1, range=[0, 1])
fig_final.update_yaxes(title_text="Score", row=2, col=2, range=[0, 1])

fig_final.update_layout(
    height=900,
    width=1400,
    title_text=f"<b>Final HGNN Model Training - {len(final_history['train_loss'])} Epochs</b>",
    showlegend=True,
    template='plotly_white'
)

fig_final.show()

# Save visualization
final_viz_path = final_model_dir / f"final_model_training_{timestamp_final}.html"
fig_final.write_html(str(final_viz_path))
print(f"✅ Final training visualization saved: {final_viz_path.name}")

# Create performance comparison with initial models (Base, GNN, HGNN initial)
print("\n📊 Creating comparison with initial models...")

# Try to load from saved CSV results if variables not in memory
comparison_dir = checkpoint_base_dir / "model_comparison"
comparison_csv = comparison_dir / "model_comparison_test_only_*.csv"

comparison_models = ['HGNN (Final)']
comparison_f1 = [final_test_metrics['f1']]
comparison_roc = [final_test_metrics['roc_auc']]

# Try to get initial model results from memory or CSV
if 'base_best_metrics' in dir() and base_best_metrics:
    comparison_models.insert(0, 'Base')
    comparison_f1.insert(0, base_best_metrics.get('f1', 0))
    comparison_roc.insert(0, base_best_metrics.get('roc_auc', 0))

if 'gnn_best_metrics' in dir() and gnn_best_metrics:
    comparison_models.insert(1 if 'Base' in comparison_models else 0, 'GNN')
    idx = 1 if 'Base' in comparison_models else 0
    comparison_f1.insert(idx, gnn_best_metrics.get('f1', 0))
    comparison_roc.insert(idx, gnn_best_metrics.get('roc_auc', 0))

if 'hgnn_best_metrics' in dir() and hgnn_best_metrics:
    comparison_models.insert(-1, 'HGNN (Initial)')
    comparison_f1.insert(-1, hgnn_best_metrics.get('f1', 0))
    comparison_roc.insert(-1, hgnn_best_metrics.get('roc_auc', 0))
else:
    # Try to load from CSV
    try:
        csv_files = list(comparison_dir.glob("model_comparison_test_only_*.csv"))
        if csv_files:
            comparison_df = pd.read_csv(csv_files[-1])  # Get most recent
            for model_name in ['Base', 'GNN', 'HGNN']:
                if model_name in comparison_df['model'].values:
                    row = comparison_df[comparison_df['model'] == model_name].iloc[0]
                    display_name = f"{model_name} (Initial)" if model_name == 'HGNN' else model_name
                    if display_name not in comparison_models:
                        comparison_models.insert(-1, display_name)
                        comparison_f1.insert(-1, float(row['f1']))
                        comparison_roc.insert(-1, float(row['roc_auc']))
            print(f"   Loaded initial model results from: {csv_files[-1].name}")
    except Exception as e:
        print(f"   ⚠️  Could not load initial model results: {e}")
        print(f"   Only showing HGNN Final results")

fig_comparison_final = go.Figure()

# Assign colors based on number of models
colors_f1 = ['lightblue', 'lightcoral', 'lightyellow', 'darkgreen'][:len(comparison_models)]
colors_roc = ['skyblue', 'salmon', 'khaki', 'forestgreen'][:len(comparison_models)]

fig_comparison_final.add_trace(go.Bar(
    x=comparison_models,
    y=comparison_f1,
    name='F1 Score',
    marker_color=colors_f1,
    text=[f'{v:.4f}' for v in comparison_f1],
    textposition='outside'
))

fig_comparison_final.add_trace(go.Bar(
    x=comparison_models,
    y=comparison_roc,
    name='ROC-AUC',
    marker_color=colors_roc,
    text=[f'{v:.4f}' for v in comparison_roc],
    textposition='outside'
))

fig_comparison_final.update_layout(
    title="<b>Model Performance Comparison: Base → GNN → HGNN → Final HGNN</b>",
    xaxis_title="Model",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.1]),
    barmode='group',
    height=600,
    width=1000,
    template='plotly_white',
    showlegend=True
)

fig_comparison_final.show()

comparison_viz_path = final_model_dir / f"model_comparison_final_{timestamp_final}.html"
fig_comparison_final.write_html(str(comparison_viz_path))
print(f"✅ Model comparison visualization saved: {comparison_viz_path.name}")

print("\n✅ All visualizations created and saved!")


📊 Creating final model visualizations...


✅ Final training visualization saved: final_model_training_20260129_090704.html

📊 Creating comparison with initial models...


✅ Model comparison visualization saved: model_comparison_final_20260129_090704.html

✅ All visualizations created and saved!
